In [188]:
# ============================================================================
# Standard Library Imports
# ============================================================================
import random
import pprint

# ============================================================================
# Third-Party Library Imports
# ============================================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ============================================================================
# Scikit-Learn Core Imports
# ============================================================================
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

# ============================================================================
# Scikit-Learn Preprocessing Imports
# ============================================================================
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

# ============================================================================
# Scikit-Learn Metrics Imports
# ============================================================================
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
)

# ============================================================================
# Scikit-Learn Model Imports
# ============================================================================
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import AdaBoostClassifier
from catboost import CatBoostClassifier
from sklearn.tree import DecisionTreeClassifier

# ============================================================================
# Constants and Configuration
# ============================================================================
RANDOM_STATE = 777
TEST_SIZE = 0.2

# Source - https://stackoverflow.com/a/9031848
# Posted by astrofrog, modified by community. See post 'Timeline' for change history
# Retrieved 2026-06-26, License - CC BY-SA 4.0

import warnings
warnings.filterwarnings('ignore')

In [189]:
df = pd.read_csv("spambase_csv.csv")

In [190]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 4601 entries, 0 to 4600
Data columns (total 58 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   word_freq_make              4601 non-null   float64
 1   word_freq_address           4601 non-null   float64
 2   word_freq_all               4601 non-null   float64
 3   word_freq_3d                4601 non-null   float64
 4   word_freq_our               4601 non-null   float64
 5   word_freq_over              4601 non-null   float64
 6   word_freq_remove            4601 non-null   float64
 7   word_freq_internet          4601 non-null   float64
 8   word_freq_order             4601 non-null   float64
 9   word_freq_mail              4601 non-null   float64
 10  word_freq_receive           4601 non-null   float64
 11  word_freq_will              4601 non-null   float64
 12  word_freq_people            4601 non-null   float64
 13  word_freq_report            4601 non-null   

In [191]:
NUMBER_OF_FEATURES = len(df.columns)

In [192]:
def print_basic_metrics(model_name: str, y_test, y_pred) -> None:
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, average='binary')
    recall = recall_score(y_test, y_pred, average='weighted')
    f1 = f1_score(y_test, y_pred, average='weighted')
    cm = confusion_matrix(y_test, y_pred)

    print("="*50)
    print(f"{model_name.upper()} PERFORMANCE METRICS")
    print("="*50)
    print(f"Accuracy:                {accuracy:.3f}")
    print(f"Precision (weighted):    {precision:.3f}")
    print(f"Recall (weighted):       {recall:.3f}")
    print(f"F1-Score (weighted):     {f1:.3f}")
    print("="*50)

    print("\nConfusion Matrix:")
    print("-----------------")
    print(f"True Negatives:  {cm[0,0]}")
    print(f"False Positives: {cm[0,1]}")
    print(f"False Negatives: {cm[1,0]}")
    print(f"True Positives:  {cm[1,1]}")
    print("="*50)

    print("\nClassification Report:")
    print(classification_report(y_test, y_pred))

In [193]:
classifier_metrics = dict()

### Logistic Regression

In [194]:
LOGREGRESSION_MODEL_NAME = "Logistic Regression"

In [195]:
numerical_features = df.drop(columns=["class"]).select_dtypes(include=np.number).columns
categorical_features = (
	df
	.drop(columns=["class"])
	.select_dtypes(include=["string", "object"])
	.columns
)

X = df.drop(columns=["class"])
y = df["class"]

X_train, X_test, y_train, y_test = train_test_split(  
	X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE  
)

### Logistic Regression Pipeline Implementation

In [196]:
logistic_regression = LogisticRegression(
    C=0.7,
    solver="lbfgs",
    max_iter=5000,
    l1_ratio=0,
)


num_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

cat_pipeline = Pipeline([  
	("imputer", SimpleImputer(strategy="most_frequent")),  
	("encoder", OneHotEncoder(handle_unknown="ignore"))  
])  

preprocessor = ColumnTransformer([
    ("num", num_pipeline, numerical_features),
    ("cat", cat_pipeline, categorical_features)
])

logistic_regression_pipeline = Pipeline(steps=[
    ("preprocessing", preprocessor),
    ("logregression_classifier", logistic_regression)
])

logistic_regression_pipeline.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessing', ...), ('logregression_classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the diffe

In [197]:
y_pred = logistic_regression_pipeline.predict(X_test)
logregression_accuracy = accuracy_score(y_test, y_pred)
classifier_metrics[f"{LOGREGRESSION_MODEL_NAME} Classifier"] = {
    "accuracy": logregression_accuracy,
    "no_features": 58,
}
print_basic_metrics(model_name=f"{LOGREGRESSION_MODEL_NAME} Classifier", y_test=y_test, y_pred=y_pred)

LOGISTIC REGRESSION CLASSIFIER PERFORMANCE METRICS
Accuracy:                0.914
Precision (weighted):    0.925
Recall (weighted):       0.914
F1-Score (weighted):     0.914

Confusion Matrix:
-----------------
True Negatives:  508
False Positives: 27
False Negatives: 52
True Positives:  334

Classification Report:
              precision    recall  f1-score   support

           0       0.91      0.95      0.93       535
           1       0.93      0.87      0.89       386

    accuracy                           0.91       921
   macro avg       0.92      0.91      0.91       921
weighted avg       0.91      0.91      0.91       921



#### Search for the Best Combination of Hyperparameters for Logistic Regression Classifier

In [198]:
param_grid = {
    "logregression_classifier__C": np.linspace(start=0.001, stop=1, num=30),
    "logregression_classifier__solver": ['lbfgs', 'newton-cg', 'newton-cholesky', 'sag', 'saga'],
    "logregression_classifier__max_iter": [num for num in range(100, 5000, 250)]
}

random_logistic_regression = RandomizedSearchCV(
    logistic_regression_pipeline,
    param_distributions=param_grid,
    n_iter=10,                       # NOTE: Number of random combinations
    cv=5,
    scoring="accuracy",
    n_jobs=-1,
    random_state=RANDOM_STATE
)

random_logistic_regression.fit(X_train, y_train)

,"estimator estimator: estimator objectAn object of that type is instantiated for each grid point.This is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step..._iter=5000))])
,"param_distributions param_distributions: dict or list of dictsDictionary with parameters names (`str`) as keys and distributionsor lists of parameters to try. Distributions must provide a ``rvs``method for sampling (such as those from scipy.stats.distributions).If a list is given, it is sampled uniformly.If a list of dicts is given, first a dict is sampled uniformly, andthen a parameter is sampled using that dict as above.","{'logregression_classifier__C': array([0.001 ..., 1. ]), 'logregression_classifier__max_iter': [100, 350, ...], 'logregression_classifier__solver': ['lbfgs', 'newton-cg', ...]}"
,"n_iter n_iter: int, default=10Number of parameter settings that are sampled. n_iter tradesoff runtime vs quality of the solution.",10
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.If None, the estimator's score method is used.",'accuracy'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given the ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``RandomizedSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscros

In [199]:
best_logreg = random_logistic_regression.best_estimator_  
classifier = best_logreg.named_steps["logregression_classifier"]

print("Best Parameters:", random_logistic_regression.best_params_)
print("Best CV Score:", random_logistic_regression.best_score_)

y_pred = best_logreg.predict(X_test)
tuned_logreg_accuracy = accuracy_score(y_test, y_pred)
print_basic_metrics(model_name=f"Tuned {LOGREGRESSION_MODEL_NAME} Classifier", y_test=y_test, y_pred=y_pred)

Best Parameters: {'logregression_classifier__solver': 'newton-cholesky', 'logregression_classifier__max_iter': 4350, 'logregression_classifier__C': np.float64(0.8966551724137931)}
Best CV Score: 0.9228260869565217
TUNED LOGISTIC REGRESSION CLASSIFIER PERFORMANCE METRICS
Accuracy:                0.913
Precision (weighted):    0.920
Recall (weighted):       0.913
F1-Score (weighted):     0.913

Confusion Matrix:
-----------------
True Negatives:  506
False Positives: 29
False Negatives: 51
True Positives:  335

Classification Report:
              precision    recall  f1-score   support

           0       0.91      0.95      0.93       535
           1       0.92      0.87      0.89       386

    accuracy                           0.91       921
   macro avg       0.91      0.91      0.91       921
weighted avg       0.91      0.91      0.91       921



In [200]:
classifier_metrics[f"Tuned {LOGREGRESSION_MODEL_NAME} Classifier"] = {
    "accuracy": tuned_logreg_accuracy,
    "no_features": 58
}

pprint.pprint(classifier_metrics)

{'Logistic Regression Classifier': {'accuracy': 0.9142236699239956,
                                    'no_features': 58},
 'Tuned Logistic Regression Classifier': {'accuracy': 0.9131378935939196,
                                          'no_features': 58}}


#### Optimizing Logistic Regression Classifier

In [201]:
importances = np.abs(classifier.coef_[0])
feature_importance = pd.Series(importances, index=X_train.columns)
feature_importance = feature_importance.sort_values(ascending=False)  
print(feature_importance)

word_freq_george              3.783025
word_freq_hp                  2.334188
word_freq_cs                  1.618136
char_freq_%24                 1.397661
word_freq_meeting             1.388354
word_freq_hpl                 1.181199
word_freq_85                  1.101690
word_freq_edu                 1.055998
word_freq_project             1.019115
word_freq_lab                 1.011156
word_freq_remove              0.957975
capital_run_length_longest    0.936001
word_freq_re                  0.889254
word_freq_conference          0.808751
word_freq_000                 0.806284
word_freq_free                0.802729
char_freq_%23                 0.763779
word_freq_3d                  0.747370
word_freq_credit              0.453785
word_freq_data                0.441557
word_freq_business            0.440138
capital_run_length_total      0.391154
word_freq_our                 0.389841
capital_run_length_average    0.373833
word_freq_technology          0.338398
char_freq_%3B            

In [206]:
important_features_scores = feature_importance[feature_importance > 0.9]
important_features = feature_importance[feature_importance > 0.9].index.to_list()
print(important_features_scores)
print(f"NUMBER OF FEATURES: {len(important_features)}")

word_freq_george              3.783025
word_freq_hp                  2.334188
word_freq_cs                  1.618136
char_freq_%24                 1.397661
word_freq_meeting             1.388354
word_freq_hpl                 1.181199
word_freq_85                  1.101690
word_freq_edu                 1.055998
word_freq_project             1.019115
word_freq_lab                 1.011156
word_freq_remove              0.957975
capital_run_length_longest    0.936001
dtype: float64
NUMBER OF FEATURES: 12


In [207]:
X_train_optimized = X_train[important_features]
X_test_optimized = X_test[important_features]

numerical_features = (
    df
    .drop(columns=["class"])
    .loc[:, important_features]
    .select_dtypes(include=np.number)
    .columns
)

categorical_features = (
	df
	.drop(columns=["class"])
	.select_dtypes(include=["string", "object"])
	.columns
)

In [ ]:
# Best Parameters: 
# {'logregression_classifier__solver': 'newton-cholesky', 
# 'logregression_classifier__max_iter': 4350, 
# 'logregression_classifier__C': np.float64(0.8966551724137931)}


In [209]:
logistic_regression = LogisticRegression(
    C=0.8966551724137931,  
    solver='newton-cholesky',  
    max_iter=4350,  
    random_state=RANDOM_STATE,
    n_jobs=-1  
)

num_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

cat_pipeline = Pipeline([  
	("imputer", SimpleImputer(strategy="most_frequent")),  
	("encoder", OneHotEncoder(handle_unknown="ignore"))  
])  

preprocessor = ColumnTransformer([
    ("num", num_pipeline, numerical_features),
    ("cat", cat_pipeline, categorical_features)
])

logistic_regression_pipeline = Pipeline(steps=[
    ("preprocessing", preprocessor),
    ("logregression_classifier", logistic_regression)
])

logistic_regression_pipeline.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessing', ...), ('logregression_classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the diffe

In [211]:
y_pred = logistic_regression_pipeline.predict(X_test)
logregression_accuracy = accuracy_score(y_test, y_pred)
print_basic_metrics(
    model_name=f"Tuned and Optimized {LOGREGRESSION_MODEL_NAME} Classifier", 
    y_test=y_test, 
    y_pred=y_pred
)

TUNED AND OPTIMIZED LOGISTIC REGRESSION CLASSIFIER PERFORMANCE METRICS
Accuracy:                0.863
Precision (weighted):    0.919
Recall (weighted):       0.863
F1-Score (weighted):     0.860

Confusion Matrix:
-----------------
True Negatives:  510
False Positives: 25
False Negatives: 101
True Positives:  285

Classification Report:
              precision    recall  f1-score   support

           0       0.83      0.95      0.89       535
           1       0.92      0.74      0.82       386

    accuracy                           0.86       921
   macro avg       0.88      0.85      0.85       921
weighted avg       0.87      0.86      0.86       921



In [212]:
classifier_metrics[f"Tuned and Optimized {LOGREGRESSION_MODEL_NAME} Classifier"] = {
    "accuracy": logregression_accuracy,
    "no_features": len(important_features_scores)
}

In [213]:
pprint.pprint(sorted(classifier_metrics.items(), key=lambda stats: stats[1]["accuracy"], reverse=True))

[('Logistic Regression Classifier',
  {'accuracy': 0.9142236699239956, 'no_features': 58}),
 ('Tuned Logistic Regression Classifier',
  {'accuracy': 0.9131378935939196, 'no_features': 58}),
 ('Tuned and Optimized Logistic Regression Classifier',
  {'accuracy': 0.8631921824104235, 'no_features': 12})]


#### Endnotes

The Logistic Regression classifier's journey through our experiments is a cautionary tale about the dangers of over-simplification, revealing that this linear model operates under fundamentally different rules than its tree-based counterparts. We began with the baseline Logistic Regression Classifier, which used all 58 features and achieved a respectable accuracy of 91.4%. For a linear model, this was a solid performance, demonstrating that even simple algorithms could extract meaningful patterns from our dataset.

Our first optimization step was the Tuned Logistic Regression Classifier. By adjusting hyperparameters like regularization strength while retaining all 58 features, we expected improvement. Instead, we witnessed a slight decline to 91.3%—a marginal drop of 0.1 percentage points. This suggested that the default parameters were already well-suited to the data, and tuning offered little benefit while potentially increasing variance through less optimal regularization.

The final iteration, the Tuned and Optimized Logistic Regression Classifier, took a dramatically different approach. We reduced the feature set from 58 down to just 12 features, implementing feature selection to eliminate redundant predictors. The result was catastrophic: accuracy plummeted to 86.3%—a staggering loss of over 5 percentage points from the baseline.

Was this tradeoff worth it? The answer is an emphatic no. Unlike tree-based models like Random Forest or XGBoost that can handle feature redundancy gracefully, Logistic Regression relies heavily on all available signals. Removing 46 features stripped away critical predictive information that couldn't be compensated by the remaining variables. The dramatic accuracy drop far outweighs any gains in simplicity or speed. This model became less complex but at an unacceptable cost to performance.

### KNN

In [214]:
KNN_MODEL_NAME = "K-Nearest Neighbors"

In [215]:
numerical_features = df.drop(columns=["class"]).select_dtypes(include=np.number).columns
categorical_features = (
	df
	.drop(columns=["class"])
	.select_dtypes(include=["string", "object"])
	.columns
)

X = df.drop(columns=["class"])
y = df["class"]

X_train, X_test, y_train, y_test = train_test_split(  
	X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE  
)

In [216]:
knn_base_classifier = KNeighborsClassifier(
    n_neighbors=5,
    weights="distance",
    metric="minkowski",
    p=2
)

num_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

cat_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", num_pipeline, numerical_features),
    ("cat", cat_pipeline, categorical_features)
])

knn_pipeline = Pipeline([
    ("preprocessing", preprocessor),
    ("knn_classifier", KNeighborsClassifier())
])

knn_pipeline.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessing', ...), ('knn_classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different trans

In [217]:
y_pred = knn_pipeline.predict(X_test)
knn_accuracy = accuracy_score(y_test, y_pred)
classifier_metrics[f"{KNN_MODEL_NAME} Classifier"] = {
    "accuracy": knn_accuracy,
    "no_features": 58,
}
print_basic_metrics(model_name=f"{KNN_MODEL_NAME} Classifier", y_test=y_test, y_pred=y_pred)

K-NEAREST NEIGHBORS CLASSIFIER PERFORMANCE METRICS
Accuracy:                0.894
Precision (weighted):    0.916
Recall (weighted):       0.894
F1-Score (weighted):     0.893

Confusion Matrix:
-----------------
True Negatives:  506
False Positives: 29
False Negatives: 69
True Positives:  317

Classification Report:
              precision    recall  f1-score   support

           0       0.88      0.95      0.91       535
           1       0.92      0.82      0.87       386

    accuracy                           0.89       921
   macro avg       0.90      0.88      0.89       921
weighted avg       0.90      0.89      0.89       921



#### Search for the Best Combination of Hyperparameters for KNN Classifier

In [218]:
param_grid = {
    "knn_classifier__n_neighbors": list(range(3, 25)),
    "knn_classifier__weights": ["uniform", "distance"],
    "knn_classifier__p": [1, 2],
    "knn_classifier__algorithm": ['auto', 'ball_tree', 'kd_tree', 'brute']
}

random_knn = RandomizedSearchCV(
    knn_pipeline,
    param_distributions=param_grid,
    n_iter=5,                   # NOTE: Number of random combinations
    cv=5,
    scoring="accuracy",
    n_jobs=-1,
    random_state=42
)

random_knn.fit(X_train, y_train)

,"estimator estimator: estimator objectAn object of that type is instantiated for each grid point.This is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...lassifier())])
,"param_distributions param_distributions: dict or list of dictsDictionary with parameters names (`str`) as keys and distributionsor lists of parameters to try. Distributions must provide a ``rvs``method for sampling (such as those from scipy.stats.distributions).If a list is given, it is sampled uniformly.If a list of dicts is given, first a dict is sampled uniformly, andthen a parameter is sampled using that dict as above.","{'knn_classifier__algorithm': ['auto', 'ball_tree', ...], 'knn_classifier__n_neighbors': [3, 4, ...], 'knn_classifier__p': [1, 2], 'knn_classifier__weights': ['uniform', 'distance']}"
,"n_iter n_iter: int, default=10Number of parameter settings that are sampled. n_iter tradesoff runtime vs quality of the solution.",5
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.If None, the estimator's score method is used.",'accuracy'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given the ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``RandomizedSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variou

In [219]:
best_knn = random_knn.best_estimator_  
classifier = best_knn.named_steps["knn_classifier"]

print("Best Parameters:", random_knn.best_params_)
print("Best CV Score:", random_knn.best_score_)

y_pred = best_knn.predict(X_test)
tuned_knn_accuracy = accuracy_score(y_test, y_pred)
print_basic_metrics(model_name=f"Tuned {KNN_MODEL_NAME} Classifier", y_test=y_test, y_pred=y_pred)

Best Parameters: {'knn_classifier__weights': 'distance', 'knn_classifier__p': 2, 'knn_classifier__n_neighbors': 16, 'knn_classifier__algorithm': 'auto'}
Best CV Score: 0.9184782608695652
TUNED K-NEAREST NEIGHBORS CLASSIFIER PERFORMANCE METRICS
Accuracy:                0.915
Precision (weighted):    0.928
Recall (weighted):       0.915
F1-Score (weighted):     0.915

Confusion Matrix:
-----------------
True Negatives:  509
False Positives: 26
False Negatives: 52
True Positives:  334

Classification Report:
              precision    recall  f1-score   support

           0       0.91      0.95      0.93       535
           1       0.93      0.87      0.90       386

    accuracy                           0.92       921
   macro avg       0.92      0.91      0.91       921
weighted avg       0.92      0.92      0.91       921



In [220]:
classifier_metrics[f"Tuned {KNN_MODEL_NAME} Classifier"] = {
    "accuracy": tuned_knn_accuracy,
    "no_features": 58
}

pprint.pprint(classifier_metrics)

{'K-Nearest Neighbors Classifier': {'accuracy': 0.8935939196525515,
                                    'no_features': 58},
 'Logistic Regression Classifier': {'accuracy': 0.9142236699239956,
                                    'no_features': 58},
 'Tuned K-Nearest Neighbors Classifier': {'accuracy': 0.9153094462540716,
                                          'no_features': 58},
 'Tuned Logistic Regression Classifier': {'accuracy': 0.9131378935939196,
                                          'no_features': 58},
 'Tuned and Optimized Logistic Regression Classifier': {'accuracy': 0.8631921824104235,
                                                        'no_features': 12}}


#### Endnotes

The K-Nearest Neighbors classifier's journey through our experiments presents a unique perspective on the bias-variance tradeoff, demonstrating that for distance-based algorithms, more data dimensions are not always beneficial. We began with the baseline **K-Nearest Neighbors Classifier**, which used all 58 features and achieved an accuracy of **89.4%**. This was a modest starting point, suggesting that the curse of dimensionality might be affecting the model's ability to find meaningful neighbors in such high-dimensional space.

Our first optimization step was the **Tuned K-Nearest Neighbors Classifier**. By adjusting hyperparameters like the number of neighbors (k) and distance metrics while retaining all 58 features, we achieved a notable improvement to **91.5%**—a gain of over 2 percentage points. This demonstrated that careful tuning could reduce bias and help the model better navigate the high-dimensional feature space, despite the inherent challenges of working with 58 features.

Interestingly, we did not pursue a simplified version of KNN with reduced features, unlike many other models in our experiments. The tuning process alone proved sufficient to unlock significant performance gains without resorting to feature reduction. This decision was validated by the results: the tuned KNN outperformed both the tuned Logistic Regression (91.3%) and even surpassed the baseline Logistic Regression (91.4%). 

When we examine the broader competitive landscape, the KNN results are revealing. The tuned KNN at **91.5%** stands as a solid mid-tier performer. It comfortably outperforms the base KNN (89.4%) and the tuned Logistic Regression (91.3%), demonstrating that proper hyperparameter optimization can substantially improve a simple yet powerful algorithm. However, it still falls short of the best-performing models, which hover in the 95% range. 

The KNN journey teaches us that sometimes, tuning existing parameters can yield significant improvements without the need for aggressive feature reduction. While dimensionality remains a challenge for KNN, the tuned version shows that with the right configuration, this distance-based algorithm can still deliver competitive results, making it a viable option for problems where interpretability and simplicity are valued over marginal gains in accuracy.

### Random Forest Classifier

In [224]:
RF_MODEL_NAME = "Random Forest"

#### Random Forest Pipeline

In [222]:
numerical_features = df.drop(columns=["class"]).select_dtypes(include=np.number).columns
categorical_features = (
	df
	.drop(columns=["class"])
	.select_dtypes(include=["string", "object"])
	.columns
)

X = df.drop(columns=["class"])
y = df["class"]

X_train, X_test, y_train, y_test = train_test_split(  
	X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE  
)

In [223]:
rf = RandomForestClassifier(
	n_estimators=500,
	max_depth=20, 
	min_samples_split=5,
	min_samples_leaf=2, 
	max_features="sqrt",
	bootstrap=True, 
	n_jobs=-1, 
	random_state=RANDOM_STATE
)

num_pipeline = Pipeline([  
	("imputer", SimpleImputer(strategy="median"))  
])  
  
cat_pipeline = Pipeline([  
	("imputer", SimpleImputer(strategy="most_frequent")),  
	("encoder", OneHotEncoder(handle_unknown="ignore"))  
])  
  
preprocessor = ColumnTransformer([  
	("num", num_pipeline, numerical_features),  
	("cat", cat_pipeline, categorical_features)  
])  
  
random_forest_pipeline = Pipeline([  
	("preprocessing", preprocessor),  
	("rf_classifier", rf)
])

random_forest_pipeline.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessing', ...), ('rf_classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transf

In [225]:
y_pred = random_forest_pipeline.predict(X_test)
rf_accuracy = accuracy_score(y_test, y_pred)
classifier_metrics[f"{RF_MODEL_NAME} Classifier"] = {
    "accuracy": rf_accuracy,
    "no_features": 58,
}
print_basic_metrics(model_name=f"{RF_MODEL_NAME} Classifier", y_test=y_test, y_pred=y_pred)

RANDOM FOREST CLASSIFIER PERFORMANCE METRICS
Accuracy:                0.933
Precision (weighted):    0.943
Recall (weighted):       0.933
F1-Score (weighted):     0.932

Confusion Matrix:
-----------------
True Negatives:  514
False Positives: 21
False Negatives: 41
True Positives:  345

Classification Report:
              precision    recall  f1-score   support

           0       0.93      0.96      0.94       535
           1       0.94      0.89      0.92       386

    accuracy                           0.93       921
   macro avg       0.93      0.93      0.93       921
weighted avg       0.93      0.93      0.93       921



In [226]:
pprint.pprint(classifier_metrics)

{'K-Nearest Neighbors Classifier': {'accuracy': 0.8935939196525515,
                                    'no_features': 58},
 'Logistic Regression Classifier': {'accuracy': 0.9142236699239956,
                                    'no_features': 58},
 'Random Forest Classifier': {'accuracy': 0.9326818675352877,
                              'no_features': 58},
 'Tuned K-Nearest Neighbors Classifier': {'accuracy': 0.9153094462540716,
                                          'no_features': 58},
 'Tuned Logistic Regression Classifier': {'accuracy': 0.9131378935939196,
                                          'no_features': 58},
 'Tuned and Optimized Logistic Regression Classifier': {'accuracy': 0.8631921824104235,
                                                        'no_features': 12}}


#### Search for the Best Combination of Hyperparameters for Random Forest Classifier

In [35]:
param_dist = {  
	"classifier__n_estimators": [100, 200, 500],  
	"classifier__max_depth": [None, 10, 20, 30],  
	"classifier__min_samples_split": [2, 5, 10],  
	"classifier__min_samples_leaf": [1, 2, 4],  
	"classifier__max_features": ["sqrt", "log2"]  
}  
  
random_random_forest = RandomizedSearchCV(  
	random_forest_pipeline,  
	param_distributions=param_dist,  
	n_iter=20,  
	cv=5,  
	scoring="accuracy",  
	n_jobs=-1,  
	random_state=42  
)  
  
random_random_forest.fit(X_train, y_train)  

,"estimator estimator: estimator objectAn object of that type is instantiated for each grid point.This is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step..._state=777))])
,"param_distributions param_distributions: dict or list of dictsDictionary with parameters names (`str`) as keys and distributionsor lists of parameters to try. Distributions must provide a ``rvs``method for sampling (such as those from scipy.stats.distributions).If a list is given, it is sampled uniformly.If a list of dicts is given, first a dict is sampled uniformly, andthen a parameter is sampled using that dict as above.","{'classifier__max_depth': [None, 10, ...], 'classifier__max_features': ['sqrt', 'log2'], 'classifier__min_samples_leaf': [1, 2, ...], 'classifier__min_samples_split': [2, 5, ...], ...}"
,"n_iter n_iter: int, default=10Number of parameter settings that are sampled. n_iter tradesoff runtime vs quality of the solution.",20
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.If None, the estimator's score method is used.",'accuracy'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given the ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``RandomizedSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the var

In [36]:
best_random_forest = random_random_forest.best_estimator_  
classifer = best_random_forest.named_steps["classifier"]

print("Best Parameters:", random_random_forest.best_params_)
print("Best CV Score:", random_random_forest.best_score_)

y_pred = best_random_forest.predict(X_test)
tuned_rf_accuracy = accuracy_score(y_test, y_pred)
print_basic_metrics(model_name="Tuned Random Forest Classifier", y_test=y_test, y_pred=y_pred)

Best Parameters: {'classifier__n_estimators': 200, 'classifier__min_samples_split': 2, 'classifier__min_samples_leaf': 1, 'classifier__max_features': 'log2', 'classifier__max_depth': 30}
Best CV Score: 0.9538043478260869
TUNED RANDOM FOREST CLASSIFIER PERFORMANCE METRICS
Accuracy:                0.946
Precision (weighted):    0.952
Recall (weighted):       0.946
F1-Score (weighted):     0.946

Confusion Matrix:
-----------------
True Negatives:  517
False Positives: 18
False Negatives: 32
True Positives:  354

Classification Report:
              precision    recall  f1-score   support

           0       0.94      0.97      0.95       535
           1       0.95      0.92      0.93       386

    accuracy                           0.95       921
   macro avg       0.95      0.94      0.94       921
weighted avg       0.95      0.95      0.95       921



In [37]:
classifier_metrics["Tuned Random Forest"] = {
    "accuracy": tuned_rf_accuracy,
    "no_features": 58
}

pprint.pprint(classifier_metrics)

{'Random Forest Base Pipeline': {'accuracy': 0.9326818675352877,
                                 'no_features': 58},
 'Tuned Random Forest': {'accuracy': 0.9457111834961998, 'no_features': 58}}


#### Optimizing Random Forest

In [38]:
importances = np.abs(classifer.feature_importances_)
feature_importance = pd.Series(importances, index=X_train.columns)
feature_importance = feature_importance.sort_values(ascending=False)  
print(feature_importance)

char_freq_%21                 0.107670
capital_run_length_average    0.075169
char_freq_%24                 0.071830
word_freq_free                0.064465
word_freq_remove              0.060242
capital_run_length_longest    0.054236
word_freq_your                0.051724
capital_run_length_total      0.050939
word_freq_hp                  0.042302
word_freq_money               0.038980
word_freq_you                 0.034430
word_freq_our                 0.032908
word_freq_000                 0.026559
word_freq_hpl                 0.020101
word_freq_george              0.017617
word_freq_edu                 0.015870
word_freq_1999                0.014950
word_freq_receive             0.013988
word_freq_internet            0.013340
char_freq_%28                 0.013049
word_freq_over                0.012236
word_freq_will                0.011734
word_freq_all                 0.011664
word_freq_mail                0.011597
word_freq_email               0.010547
word_freq_re             

In [39]:
important_features_scores = feature_importance[feature_importance > 0.05]
important_features = feature_importance[feature_importance > 0.05].index.to_list()
print(important_features_scores)

char_freq_%21                 0.107670
capital_run_length_average    0.075169
char_freq_%24                 0.071830
word_freq_free                0.064465
word_freq_remove              0.060242
capital_run_length_longest    0.054236
word_freq_your                0.051724
capital_run_length_total      0.050939
dtype: float64


In [40]:
X_train_optimized = X_train[important_features]
X_test_optimized = X_test[important_features]

numerical_features = (
    df
    .drop(columns=["class"])
    .loc[:, important_features]
    .select_dtypes(include=np.number)
    .columns
)

In [41]:
rf = RandomForestClassifier(
	n_estimators=500,
	max_depth=20, 
	min_samples_split=5,
	min_samples_leaf=2, 
	max_features="sqrt",
	bootstrap=True, 
	n_jobs=-1, 
	random_state=RANDOM_STATE
)

num_pipeline = Pipeline([  
	("imputer", SimpleImputer(strategy="median"))  
])  
  
cat_pipeline = Pipeline([  
	("imputer", SimpleImputer(strategy="most_frequent")),  
	("encoder", OneHotEncoder(handle_unknown="ignore"))  
])  
  
preprocessor = ColumnTransformer([  
	("num", num_pipeline, numerical_features),  
	("cat", cat_pipeline, categorical_features)  
])  
  
random_forest_pipeline = Pipeline([  
	("preprocessing", preprocessor),  
	("classifier", rf)
])

random_forest_pipeline.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessing', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transform

In [42]:
y_pred = random_forest_pipeline.predict(X_test)
rf_accuracy = accuracy_score(y_test, y_pred)
print_basic_metrics(
    model_name="Tuned and Optimized Random Forest Classifier", 
    y_test=y_test, 
    y_pred=y_pred
)

TUNED AND OPTIMIZED RANDOM FOREST CLASSIFIER PERFORMANCE METRICS
Accuracy:                0.908
Precision (weighted):    0.934
Recall (weighted):       0.908
F1-Score (weighted):     0.907

Confusion Matrix:
-----------------
True Negatives:  512
False Positives: 23
False Negatives: 62
True Positives:  324

Classification Report:
              precision    recall  f1-score   support

           0       0.89      0.96      0.92       535
           1       0.93      0.84      0.88       386

    accuracy                           0.91       921
   macro avg       0.91      0.90      0.90       921
weighted avg       0.91      0.91      0.91       921



In [43]:
classifier_metrics["Tuned and Optimized Random Forest Classifier"] = {
    "accuracy": rf_accuracy,
    "no_features": len(important_features_scores)
}

In [44]:
classifier_metrics

{'Random Forest Base Pipeline': {'accuracy': 0.9326818675352877,
  'no_features': 58},
 'Tuned Random Forest': {'accuracy': 0.9457111834961998, 'no_features': 58},
 'Tuned and Optimized Random Forest Classifier': {'accuracy': 0.9077090119435396,
  'no_features': 8}}

#### Random Forest vs. Other classifiers

#### Saving classifier

### Extreme Random Forest

In [45]:
numerical_features = df.drop(columns=["class"]).select_dtypes(include=np.number).columns
categorical_features = []

X = df.drop(columns=["class"])
y = df["class"]

X_train, X_test, y_train, y_test = train_test_split(  
	X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE  
)

#### Extreme Random Forest Pipeline

In [46]:
from sklearn.ensemble import ExtraTreesClassifier

In [47]:
xrf = ExtraTreesClassifier(
    n_estimators=300,
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    max_features="sqrt",
    bootstrap=False,
    criterion="gini",
    random_state=RANDOM_STATE,
    n_jobs=-1
)

In [48]:
num_pipeline = Pipeline([  
	("imputer", SimpleImputer(strategy="median"))  
])  
  
cat_pipeline = Pipeline([  
	("imputer", SimpleImputer(strategy="most_frequent")),  
	("encoder", OneHotEncoder(handle_unknown="ignore"))  
])  
  
preprocessor = ColumnTransformer([  
	("num", num_pipeline, numerical_features),  
	("cat", cat_pipeline, categorical_features)  
])  
  
extreme_random_forest_pipeline = Pipeline([  
	("preprocessing", preprocessor),  
	("xrf_classifier", xrf)
])

extreme_random_forest_pipeline.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessing', ...), ('xrf_classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different trans

In [49]:
y_pred = extreme_random_forest_pipeline.predict(X_test)
xrf_accuracy = accuracy_score(y_test, y_pred)
classifier_metrics["Extreme Random Forest Classifier"] = {
    "accuracy": xrf_accuracy,
    "no_features": 58,
}
print_basic_metrics(model_name="Extreme Random Forest Classifier", y_test=y_test, y_pred=y_pred)

EXTREME RANDOM FOREST CLASSIFIER PERFORMANCE METRICS
Accuracy:                0.955
Precision (weighted):    0.960
Recall (weighted):       0.955
F1-Score (weighted):     0.955

Confusion Matrix:
-----------------
True Negatives:  520
False Positives: 15
False Negatives: 26
True Positives:  360

Classification Report:
              precision    recall  f1-score   support

           0       0.95      0.97      0.96       535
           1       0.96      0.93      0.95       386

    accuracy                           0.96       921
   macro avg       0.96      0.95      0.95       921
weighted avg       0.96      0.96      0.96       921



In [50]:
pprint.pprint(classifier_metrics)

{'Extreme Random Forest Classifier': {'accuracy': 0.9554831704668838,
                                      'no_features': 58},
 'Random Forest Base Pipeline': {'accuracy': 0.9326818675352877,
                                 'no_features': 58},
 'Tuned Random Forest': {'accuracy': 0.9457111834961998, 'no_features': 58},
 'Tuned and Optimized Random Forest Classifier': {'accuracy': 0.9077090119435396,
                                                  'no_features': 8}}


#### Random Search of Hyperparameters for Extreme Random Forest

In [51]:
# extended hyperparameter tuning space
param_dist = {
    "xrf_classifier__n_estimators": [100, 200, 300, 500, 800],
    "xrf_classifier__criterion": ["gini", "entropy", "log_loss"],
    "xrf_classifier__max_depth": [None, 10, 20, 30, 50],
    "xrf_classifier__min_samples_split": [2, 5, 10, 20],
    "xrf_classifier__min_samples_leaf": [1, 2, 4, 8],
    "xrf_classifier__max_features": ["sqrt", "log2", None, 0.5, 0.8],
    "xrf_classifier__bootstrap": [True],
    "xrf_classifier__max_samples": [None, 0.5, 0.7, 0.9],
    "xrf_classifier__min_impurity_decrease": [0.0, 0.01, 0.05],
    "xrf_classifier__ccp_alpha": [0.0, 0.001, 0.01],
    "xrf_classifier__class_weight": [None, "balanced", "balanced_subsample"]
}

search = RandomizedSearchCV(
    extreme_random_forest_pipeline,
    param_distributions=param_dist,
    n_iter=20,
    cv=5,
    scoring="accuracy",
    n_jobs=-1,
    random_state=RANDOM_STATE,
    verbose=1
)

search.fit(X_train, y_train)

Fitting 5 folds for each of 20 candidates, totalling 100 fits


,"estimator estimator: estimator objectAn object of that type is instantiated for each grid point.This is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step..._state=777))])
,"param_distributions param_distributions: dict or list of dictsDictionary with parameters names (`str`) as keys and distributionsor lists of parameters to try. Distributions must provide a ``rvs``method for sampling (such as those from scipy.stats.distributions).If a list is given, it is sampled uniformly.If a list of dicts is given, first a dict is sampled uniformly, andthen a parameter is sampled using that dict as above.","{'xrf_classifier__bootstrap': [True], 'xrf_classifier__ccp_alpha': [0.0, 0.001, ...], 'xrf_classifier__class_weight': [None, 'balanced', ...], 'xrf_classifier__criterion': ['gini', 'entropy', ...], ...}"
,"n_iter n_iter: int, default=10Number of parameter settings that are sampled. n_iter tradesoff runtime vs quality of the solution.",20
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.If None, the estimator's score method is used.",'accuracy'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given the ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``RandomizedSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User G

In [52]:
best_extreme_random_forest = search.best_estimator_  
classifer = best_extreme_random_forest.named_steps["xrf_classifier"]

print("Best Parameters:", search.best_params_)
print("Best CV Score:", search.best_score_)

y_pred = best_extreme_random_forest.predict(X_test)
tuned_xrf_accuracy = accuracy_score(y_test, y_pred)
print_basic_metrics(model_name="Tuned Random Forest Classifier", y_test=y_test, y_pred=y_pred)

Best Parameters: {'xrf_classifier__n_estimators': 200, 'xrf_classifier__min_samples_split': 2, 'xrf_classifier__min_samples_leaf': 2, 'xrf_classifier__min_impurity_decrease': 0.0, 'xrf_classifier__max_samples': 0.5, 'xrf_classifier__max_features': 0.5, 'xrf_classifier__max_depth': 20, 'xrf_classifier__criterion': 'log_loss', 'xrf_classifier__class_weight': None, 'xrf_classifier__ccp_alpha': 0.0, 'xrf_classifier__bootstrap': True}
Best CV Score: 0.9426630434782609
TUNED RANDOM FOREST CLASSIFIER PERFORMANCE METRICS
Accuracy:                0.934
Precision (weighted):    0.940
Recall (weighted):       0.934
F1-Score (weighted):     0.934

Confusion Matrix:
-----------------
True Negatives:  513
False Positives: 22
False Negatives: 39
True Positives:  347

Classification Report:
              precision    recall  f1-score   support

           0       0.93      0.96      0.94       535
           1       0.94      0.90      0.92       386

    accuracy                           0.93       

In [53]:
classifier_metrics["Tuned Extreme Random Forest"] = {
    "accuracy": tuned_xrf_accuracy,
    "no_features": 58
}

pprint.pprint(classifier_metrics)

{'Extreme Random Forest Classifier': {'accuracy': 0.9554831704668838,
                                      'no_features': 58},
 'Random Forest Base Pipeline': {'accuracy': 0.9326818675352877,
                                 'no_features': 58},
 'Tuned Extreme Random Forest': {'accuracy': 0.9337676438653637,
                                 'no_features': 58},
 'Tuned Random Forest': {'accuracy': 0.9457111834961998, 'no_features': 58},
 'Tuned and Optimized Random Forest Classifier': {'accuracy': 0.9077090119435396,
                                                  'no_features': 8}}


#### Optimizing Extreme Random Forest

In [54]:
importances = np.abs(classifer.feature_importances_)
feature_importance = pd.Series(importances, index=X_train.columns)
feature_importance = feature_importance.sort_values(ascending=False)  
print(feature_importance)

char_freq_%24                 0.078482
word_freq_remove              0.078191
word_freq_your                0.072690
char_freq_%21                 0.070127
word_freq_hp                  0.062518
capital_run_length_longest    0.053313
word_freq_free                0.048878
word_freq_000                 0.047398
capital_run_length_average    0.036053
word_freq_george              0.033069
word_freq_our                 0.032912
word_freq_money               0.029924
word_freq_you                 0.027134
word_freq_hpl                 0.026575
capital_run_length_total      0.024649
word_freq_edu                 0.017042
word_freq_receive             0.016872
word_freq_business            0.016543
word_freq_1999                0.016066
word_freq_over                0.015618
word_freq_internet            0.015264
word_freq_all                 0.014374
word_freq_re                  0.013286
word_freq_will                0.012943
word_freq_order               0.011452
word_freq_email          

In [55]:
important_features_scores = feature_importance[feature_importance > 0.035]
important_features = feature_importance[feature_importance > 0.035].index.to_list()
print(important_features_scores)
print(f"NUMBER OF FEATURES: {len(important_features)}")

char_freq_%24                 0.078482
word_freq_remove              0.078191
word_freq_your                0.072690
char_freq_%21                 0.070127
word_freq_hp                  0.062518
capital_run_length_longest    0.053313
word_freq_free                0.048878
word_freq_000                 0.047398
capital_run_length_average    0.036053
dtype: float64
NUMBER OF FEATURES: 9


In [56]:
X_train_optimized = X_train[important_features]
X_test_optimized = X_test[important_features]

numerical_features = (
    df
    .drop(columns=["class"])
    .loc[:, important_features]
    .select_dtypes(include=np.number)
    .columns
)

In [57]:
num_pipeline = Pipeline([  
	("imputer", SimpleImputer(strategy="median"))  
])  
  
cat_pipeline = Pipeline([  
	("imputer", SimpleImputer(strategy="most_frequent")),  
	("encoder", OneHotEncoder(handle_unknown="ignore"))  
])  
  
preprocessor = ColumnTransformer([  
	("num", num_pipeline, numerical_features),  
	("cat", cat_pipeline, categorical_features)  
])  
  
extreme_random_forest_pipeline = Pipeline([  
	("preprocessing", preprocessor),  
	("xrf_classifier", xrf)
])

extreme_random_forest_pipeline.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessing', ...), ('xrf_classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different trans

In [58]:
y_pred = extreme_random_forest_pipeline.predict(X_test)
xrf_accuracy = accuracy_score(y_test, y_pred)
print_basic_metrics(
    model_name="Tuned and Optimized Extreme Random Forest Classifier", 
    y_test=y_test, 
    y_pred=y_pred
)

TUNED AND OPTIMIZED EXTREME RANDOM FOREST CLASSIFIER PERFORMANCE METRICS
Accuracy:                0.932
Precision (weighted):    0.935
Recall (weighted):       0.932
F1-Score (weighted):     0.931

Confusion Matrix:
-----------------
True Negatives:  511
False Positives: 24
False Negatives: 39
True Positives:  347

Classification Report:
              precision    recall  f1-score   support

           0       0.93      0.96      0.94       535
           1       0.94      0.90      0.92       386

    accuracy                           0.93       921
   macro avg       0.93      0.93      0.93       921
weighted avg       0.93      0.93      0.93       921



In [59]:
classifier_metrics["Tuned and Optimized Extreme Random Forest Classifier"] = {
    "accuracy": xrf_accuracy,
    "no_features": len(important_features_scores)
}

In [60]:
pprint.pprint(sorted(classifier_metrics.items(), key=lambda stats: stats[1]["accuracy"], reverse=True))

[('Extreme Random Forest Classifier',
  {'accuracy': 0.9554831704668838, 'no_features': 58}),
 ('Tuned Random Forest', {'accuracy': 0.9457111834961998, 'no_features': 58}),
 ('Tuned Extreme Random Forest',
  {'accuracy': 0.9337676438653637, 'no_features': 58}),
 ('Random Forest Base Pipeline',
  {'accuracy': 0.9326818675352877, 'no_features': 58}),
 ('Tuned and Optimized Extreme Random Forest Classifier',
  {'accuracy': 0.9315960912052117, 'no_features': 9}),
 ('Tuned and Optimized Random Forest Classifier',
  {'accuracy': 0.9077090119435396, 'no_features': 8})]


### Multilayer Perceptron

In [61]:
from sklearn.neural_network import MLPClassifier  

In [62]:
numerical_features = df.drop(columns=["class"]).select_dtypes(include=np.number).columns
categorical_features = []

X = df.drop(columns=["class"])
y = df["class"]

X_train, X_test, y_train, y_test = train_test_split(  
	X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE  
)

#### Multilayer Perceptron Pipeline

In [63]:
mlp = MLPClassifier(  
	hidden_layer_sizes=(64, 32),  
	activation="relu",  
	solver="adam",  
	max_iter=300,  
	random_state=RANDOM_STATE  
) 

num_pipeline = Pipeline([  
	("imputer", SimpleImputer(strategy="median"))  
])  
  
cat_pipeline = Pipeline([  
	("imputer", SimpleImputer(strategy="most_frequent")),  
	("encoder", OneHotEncoder(handle_unknown="ignore"))  
])  
  
preprocessor = ColumnTransformer([  
	("num", num_pipeline, numerical_features),  
	("cat", cat_pipeline, categorical_features)  
])  
  
mlp_pipeline = Pipeline([  
	("preprocessing", preprocessor),  
	("mlp_classifier", mlp)
])

mlp_pipeline.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessing', ...), ('mlp_classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different trans

In [64]:
y_pred = mlp_pipeline.predict(X_test)
mlp_accuracy = accuracy_score(y_test, y_pred)
classifier_metrics["Multilayer Perceptor Classifier"] = {
    "accuracy": mlp_accuracy,
    "no_features": 58,
}
print_basic_metrics(model_name="Multilayer Perceptor Classifier", y_test=y_test, y_pred=y_pred)

MULTILAYER PERCEPTOR CLASSIFIER PERFORMANCE METRICS
Accuracy:                0.924
Precision (weighted):    0.932
Recall (weighted):       0.924
F1-Score (weighted):     0.924

Confusion Matrix:
-----------------
True Negatives:  510
False Positives: 25
False Negatives: 45
True Positives:  341

Classification Report:
              precision    recall  f1-score   support

           0       0.92      0.95      0.94       535
           1       0.93      0.88      0.91       386

    accuracy                           0.92       921
   macro avg       0.93      0.92      0.92       921
weighted avg       0.92      0.92      0.92       921



In [65]:
pprint.pprint(classifier_metrics)

{'Extreme Random Forest Classifier': {'accuracy': 0.9554831704668838,
                                      'no_features': 58},
 'Multilayer Perceptor Classifier': {'accuracy': 0.9239956568946797,
                                     'no_features': 58},
 'Random Forest Base Pipeline': {'accuracy': 0.9326818675352877,
                                 'no_features': 58},
 'Tuned Extreme Random Forest': {'accuracy': 0.9337676438653637,
                                 'no_features': 58},
 'Tuned Random Forest': {'accuracy': 0.9457111834961998, 'no_features': 58},
 'Tuned and Optimized Extreme Random Forest Classifier': {'accuracy': 0.9315960912052117,
                                                          'no_features': 9},
 'Tuned and Optimized Random Forest Classifier': {'accuracy': 0.9077090119435396,
                                                  'no_features': 8}}


#### Random Search of Hyperparameters for Multiplayer Perceptor

In [66]:
param_dist = {  
	"mlp_classifier__hidden_layer_sizes": [(50,), (100,), (100, 50), (128, 64), (64, 32, 16)],  
	"mlp_classifier__activation": ["relu", "tanh"],  
	"mlp_classifier__solver": ["adam", "sgd"],  
	"mlp_classifier__alpha": [1e-5, 1e-4, 1e-3, 1e-2],  
	"mlp_classifier__learning_rate": ["constant", "adaptive"],  
	"mlp_classifier__learning_rate_init": [0.001, 0.01, 0.1],  
	"mlp_classifier__batch_size": [32, 64, 128],  
	"mlp_classifier__max_iter": [200, 300, 500]  
}  
  
random_search_mlp = RandomizedSearchCV(  
	mlp_pipeline,  
	param_distributions=param_dist,  
	n_iter=20,  
	cv=5,  
	scoring="accuracy",  
	n_jobs=-1,  
	verbose=2,  
	random_state=RANDOM_STATE  
)  
  
random_search_mlp.fit(X_train, y_train)  

Fitting 5 folds for each of 20 candidates, totalling 100 fits


,"estimator estimator: estimator objectAn object of that type is instantiated for each grid point.This is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step..._state=777))])
,"param_distributions param_distributions: dict or list of dictsDictionary with parameters names (`str`) as keys and distributionsor lists of parameters to try. Distributions must provide a ``rvs``method for sampling (such as those from scipy.stats.distributions).If a list is given, it is sampled uniformly.If a list of dicts is given, first a dict is sampled uniformly, andthen a parameter is sampled using that dict as above.","{'mlp_classifier__activation': ['relu', 'tanh'], 'mlp_classifier__alpha': [1e-05, 0.0001, ...], 'mlp_classifier__batch_size': [32, 64, ...], 'mlp_classifier__hidden_layer_sizes': [(50,), (100,), ...], ...}"
,"n_iter n_iter: int, default=10Number of parameter settings that are sampled. n_iter tradesoff runtime vs quality of the solution.",20
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.If None, the estimator's score method is used.",'accuracy'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given the ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``RandomizedSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`Use

#### Optimizing Multilayer Perceptor

In [67]:
best_mlp = random_search_mlp.best_estimator_  
classifer = best_mlp.named_steps["mlp_classifier"]

print("Best Parameters:", random_search_mlp.best_params_)
print("Best CV Score:", random_search_mlp.best_score_)

y_pred = best_mlp.predict(X_test)
tuned_mlp_accuracy = accuracy_score(y_test, y_pred)
print_basic_metrics(model_name="Tuned Random Forest Classifier", y_test=y_test, y_pred=y_pred)

Best Parameters: {'mlp_classifier__solver': 'adam', 'mlp_classifier__max_iter': 200, 'mlp_classifier__learning_rate_init': 0.001, 'mlp_classifier__learning_rate': 'constant', 'mlp_classifier__hidden_layer_sizes': (100, 50), 'mlp_classifier__batch_size': 128, 'mlp_classifier__alpha': 0.01, 'mlp_classifier__activation': 'tanh'}
Best CV Score: 0.9372282608695652
TUNED RANDOM FOREST CLASSIFIER PERFORMANCE METRICS
Accuracy:                0.933
Precision (weighted):    0.911
Recall (weighted):       0.933
F1-Score (weighted):     0.933

Confusion Matrix:
-----------------
True Negatives:  500
False Positives: 35
False Negatives: 27
True Positives:  359

Classification Report:
              precision    recall  f1-score   support

           0       0.95      0.93      0.94       535
           1       0.91      0.93      0.92       386

    accuracy                           0.93       921
   macro avg       0.93      0.93      0.93       921
weighted avg       0.93      0.93      0.93     

In [68]:
classifier_metrics["Tuned Multilayer Perceptor"] = {
    "accuracy": tuned_mlp_accuracy,
    "no_features": 58
}

pprint.pprint(classifier_metrics)

{'Extreme Random Forest Classifier': {'accuracy': 0.9554831704668838,
                                      'no_features': 58},
 'Multilayer Perceptor Classifier': {'accuracy': 0.9239956568946797,
                                     'no_features': 58},
 'Random Forest Base Pipeline': {'accuracy': 0.9326818675352877,
                                 'no_features': 58},
 'Tuned Extreme Random Forest': {'accuracy': 0.9337676438653637,
                                 'no_features': 58},
 'Tuned Multilayer Perceptor': {'accuracy': 0.9326818675352877,
                                'no_features': 58},
 'Tuned Random Forest': {'accuracy': 0.9457111834961998, 'no_features': 58},
 'Tuned and Optimized Extreme Random Forest Classifier': {'accuracy': 0.9315960912052117,
                                                          'no_features': 9},
 'Tuned and Optimized Random Forest Classifier': {'accuracy': 0.9077090119435396,
                                                  'no_features': 8}}

### Gradient Boosting Classifier

In [69]:
GB_MODEL_NAME = "Gradient Boosting"

In [70]:
from sklearn.ensemble import GradientBoostingClassifier

In [71]:
numerical_features = df.drop(columns=["class"]).select_dtypes(include=np.number).columns
categorical_features = []

X = df.drop(columns=["class"])
y = df["class"]

X_train, X_test, y_train, y_test = train_test_split(  
	X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE  
)

#### Gradient Boosting Base Pipeline

In [72]:
gradient_boosting = GradientBoostingClassifier(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=3,
    subsample=0.8,
    random_state=42
)

num_pipeline = Pipeline([  
	("imputer", SimpleImputer(strategy="median"))  
])  
  
cat_pipeline = Pipeline([  
	("imputer", SimpleImputer(strategy="most_frequent")),  
	("encoder", OneHotEncoder(handle_unknown="ignore"))  
])  
  
preprocessor = ColumnTransformer([  
	("num", num_pipeline, numerical_features),  
	("cat", cat_pipeline, categorical_features)  
])  
  
gb_pipeline = Pipeline([  
	("preprocessing", preprocessor),  
	("gb_classifier", gradient_boosting)
])

gb_pipeline.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessing', ...), ('gb_classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transf

In [73]:
y_pred = gb_pipeline.predict(X_test)
gb_accuracy = accuracy_score(y_test, y_pred)
classifier_metrics[f"{GB_MODEL_NAME} Classifier"] = {
    "accuracy": gb_accuracy,
    "no_features": 58,
}
print_basic_metrics(model_name=f"{GB_MODEL_NAME} Classifier", y_test=y_test, y_pred=y_pred)

GRADIENT BOOSTING CLASSIFIER PERFORMANCE METRICS
Accuracy:                0.939
Precision (weighted):    0.941
Recall (weighted):       0.939
F1-Score (weighted):     0.939

Confusion Matrix:
-----------------
True Negatives:  513
False Positives: 22
False Negatives: 34
True Positives:  352

Classification Report:
              precision    recall  f1-score   support

           0       0.94      0.96      0.95       535
           1       0.94      0.91      0.93       386

    accuracy                           0.94       921
   macro avg       0.94      0.94      0.94       921
weighted avg       0.94      0.94      0.94       921



In [74]:
classifier_metrics

{'Random Forest Base Pipeline': {'accuracy': 0.9326818675352877,
  'no_features': 58},
 'Tuned Random Forest': {'accuracy': 0.9457111834961998, 'no_features': 58},
 'Tuned and Optimized Random Forest Classifier': {'accuracy': 0.9077090119435396,
  'no_features': 8},
 'Extreme Random Forest Classifier': {'accuracy': 0.9554831704668838,
  'no_features': 58},
 'Tuned Extreme Random Forest': {'accuracy': 0.9337676438653637,
  'no_features': 58},
 'Tuned and Optimized Extreme Random Forest Classifier': {'accuracy': 0.9315960912052117,
  'no_features': 9},
 'Multilayer Perceptor Classifier': {'accuracy': 0.9239956568946797,
  'no_features': 58},
 'Tuned Multilayer Perceptor': {'accuracy': 0.9326818675352877,
  'no_features': 58},
 'Gradient Boosting Classifier': {'accuracy': 0.9391965255157437,
  'no_features': 58}}

#### Search for the Best Combination of Hyperparameters for Gradient Boosting

In [75]:
param_dist = {
    "gb_classifier__n_estimators": [100, 200, 300],
    "gb_classifier__learning_rate": [0.01, 0.05, 0.1, 0.2],
    "gb_classifier__max_depth": [3, 4, 5],
    "gb_classifier__min_samples_split": [2, 5, 10],
    "gb_classifier__min_samples_leaf": [1, 2, 4],
    "gb_classifier__subsample": [0.6, 0.8, 1.0],
    "gb_classifier__max_features": ["sqrt", "log2", None]
}

random_gb = RandomizedSearchCV(
    gb_pipeline,
    param_distributions=param_dist,
    n_iter=20,
    cv=5,
    scoring="accuracy",
    n_jobs=-1,
    random_state=RANDOM_STATE
)

random_gb.fit(X_train, y_train)

,"estimator estimator: estimator objectAn object of that type is instantiated for each grid point.This is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...sample=0.8))])
,"param_distributions param_distributions: dict or list of dictsDictionary with parameters names (`str`) as keys and distributionsor lists of parameters to try. Distributions must provide a ``rvs``method for sampling (such as those from scipy.stats.distributions).If a list is given, it is sampled uniformly.If a list of dicts is given, first a dict is sampled uniformly, andthen a parameter is sampled using that dict as above.","{'gb_classifier__learning_rate': [0.01, 0.05, ...], 'gb_classifier__max_depth': [3, 4, ...], 'gb_classifier__max_features': ['sqrt', 'log2', ...], 'gb_classifier__min_samples_leaf': [1, 2, ...], ...}"
,"n_iter n_iter: int, default=10Number of parameter settings that are sampled. n_iter tradesoff runtime vs quality of the solution.",20
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.If None, the estimator's score method is used.",'accuracy'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given the ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``RandomizedSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guid

In [76]:
best_gb = random_gb.best_estimator_  
classifer = best_gb.named_steps["gb_classifier"]

print("Best Parameters:", random_gb.best_params_)
print("Best CV Score:", random_gb.best_score_)

y_pred = best_gb.predict(X_test)
tuned_gb_accuracy = accuracy_score(y_test, y_pred)
print_basic_metrics(model_name=f"Tuned {GB_MODEL_NAME} Classifier", y_test=y_test, y_pred=y_pred)

Best Parameters: {'gb_classifier__subsample': 1.0, 'gb_classifier__n_estimators': 200, 'gb_classifier__min_samples_split': 10, 'gb_classifier__min_samples_leaf': 1, 'gb_classifier__max_features': 'log2', 'gb_classifier__max_depth': 5, 'gb_classifier__learning_rate': 0.05}
Best CV Score: 0.954891304347826
TUNED GRADIENT BOOSTING CLASSIFIER PERFORMANCE METRICS
Accuracy:                0.946
Precision (weighted):    0.944
Recall (weighted):       0.946
F1-Score (weighted):     0.946

Confusion Matrix:
-----------------
True Negatives:  514
False Positives: 21
False Negatives: 29
True Positives:  357

Classification Report:
              precision    recall  f1-score   support

           0       0.95      0.96      0.95       535
           1       0.94      0.92      0.93       386

    accuracy                           0.95       921
   macro avg       0.95      0.94      0.94       921
weighted avg       0.95      0.95      0.95       921



In [77]:
classifier_metrics[f"Tuned {GB_MODEL_NAME} Classifier"] = {
    "accuracy": tuned_gb_accuracy,
    "no_features": 58
}

pprint.pprint(classifier_metrics)

{'Extreme Random Forest Classifier': {'accuracy': 0.9554831704668838,
                                      'no_features': 58},
 'Gradient Boosting Classifier': {'accuracy': 0.9391965255157437,
                                  'no_features': 58},
 'Multilayer Perceptor Classifier': {'accuracy': 0.9239956568946797,
                                     'no_features': 58},
 'Random Forest Base Pipeline': {'accuracy': 0.9326818675352877,
                                 'no_features': 58},
 'Tuned Extreme Random Forest': {'accuracy': 0.9337676438653637,
                                 'no_features': 58},
 'Tuned Gradient Boosting Classifier': {'accuracy': 0.9457111834961998,
                                        'no_features': 58},
 'Tuned Multilayer Perceptor': {'accuracy': 0.9326818675352877,
                                'no_features': 58},
 'Tuned Random Forest': {'accuracy': 0.9457111834961998, 'no_features': 58},
 'Tuned and Optimized Extreme Random Forest Classifier': {'accura

#### Optimizing Gradient Boosting

In [78]:
importances = np.abs(classifer.feature_importances_)
feature_importance = pd.Series(importances, index=X_train.columns)
feature_importance = feature_importance.sort_values(ascending=False)  
print(feature_importance)

char_freq_%24                 0.112690
char_freq_%21                 0.099300
word_freq_hp                  0.091311
word_freq_remove              0.071437
capital_run_length_longest    0.061488
capital_run_length_average    0.059587
word_freq_free                0.057158
word_freq_your                0.048076
capital_run_length_total      0.036953
word_freq_george              0.032641
word_freq_money               0.031458
word_freq_hpl                 0.029770
word_freq_edu                 0.024488
word_freq_000                 0.023086
word_freq_our                 0.022736
word_freq_receive             0.022062
word_freq_business            0.019130
word_freq_internet            0.012659
word_freq_mail                0.011809
word_freq_1999                0.010412
word_freq_you                 0.010175
word_freq_meeting             0.008325
word_freq_all                 0.007746
word_freq_re                  0.007298
word_freq_address             0.007068
word_freq_over           

In [79]:
important_features_scores = feature_importance[feature_importance > 0.02]
important_features = feature_importance[feature_importance > 0.02].index.to_list()
print(important_features_scores)
print(f"NUMBER OF FEATURES: {len(important_features)}")

char_freq_%24                 0.112690
char_freq_%21                 0.099300
word_freq_hp                  0.091311
word_freq_remove              0.071437
capital_run_length_longest    0.061488
capital_run_length_average    0.059587
word_freq_free                0.057158
word_freq_your                0.048076
capital_run_length_total      0.036953
word_freq_george              0.032641
word_freq_money               0.031458
word_freq_hpl                 0.029770
word_freq_edu                 0.024488
word_freq_000                 0.023086
word_freq_our                 0.022736
word_freq_receive             0.022062
dtype: float64
NUMBER OF FEATURES: 16


In [80]:
X_train_optimized = X_train[important_features]
X_test_optimized = X_test[important_features]

numerical_features = (
    df
    .drop(columns=["class"])
    .loc[:, important_features]
    .select_dtypes(include=np.number)
    .columns
)

In [81]:
# Best Parameters: 
# {'gb_classifier__subsample': 1.0, 
# 'gb_classifier__n_estimators': 200, 
# 'gb_classifier__min_samples_split': 10, 
# 'gb_classifier__min_samples_leaf': 1, 
# 'gb_classifier__max_features': 'log2', 
# 'gb_classifier__max_depth': 5, 
# 'gb_classifier__learning_rate': 0.05}


In [82]:
gradient_boosting = GradientBoostingClassifier(
    subsample=1.0,
    n_estimators=200,
    min_samples_split=10,
    min_samples_leaf=1,
    max_features="log2",
    max_depth=5,
    learning_rate=0.05
)

num_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])
  
cat_pipeline = Pipeline([  
	("imputer", SimpleImputer(strategy="most_frequent")),  
	("encoder", OneHotEncoder(handle_unknown="ignore"))  
])  
  
preprocessor = ColumnTransformer([  
	("num", num_pipeline, numerical_features),  
	("cat", cat_pipeline, categorical_features)  
])  
  
gb_pipeline = Pipeline([  
	("preprocessing", preprocessor),  
	("gb_classifier", gradient_boosting)
])

gb_pipeline.fit(X_train_optimized, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessing', ...), ('gb_classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transf

In [83]:
y_pred = gb_pipeline.predict(X_test)
gb_accuracy = accuracy_score(y_test, y_pred)
print_basic_metrics(
    model_name=f"Tuned and Optimized {GB_MODEL_NAME} Classifier", 
    y_test=y_test, 
    y_pred=y_pred
)

TUNED AND OPTIMIZED GRADIENT BOOSTING CLASSIFIER PERFORMANCE METRICS
Accuracy:                0.939
Precision (weighted):    0.944
Recall (weighted):       0.939
F1-Score (weighted):     0.939

Confusion Matrix:
-----------------
True Negatives:  514
False Positives: 21
False Negatives: 35
True Positives:  351

Classification Report:
              precision    recall  f1-score   support

           0       0.94      0.96      0.95       535
           1       0.94      0.91      0.93       386

    accuracy                           0.94       921
   macro avg       0.94      0.94      0.94       921
weighted avg       0.94      0.94      0.94       921



In [84]:
classifier_metrics[f"Tuned and Optimized {GB_MODEL_NAME} Classifier"] = {
    "accuracy": gb_accuracy,
    "no_features": len(important_features_scores)
}

In [85]:
pprint.pprint(sorted(classifier_metrics.items(), key=lambda stats: stats[1]["accuracy"], reverse=True))

[('Extreme Random Forest Classifier',
  {'accuracy': 0.9554831704668838, 'no_features': 58}),
 ('Tuned Random Forest', {'accuracy': 0.9457111834961998, 'no_features': 58}),
 ('Tuned Gradient Boosting Classifier',
  {'accuracy': 0.9457111834961998, 'no_features': 58}),
 ('Gradient Boosting Classifier',
  {'accuracy': 0.9391965255157437, 'no_features': 58}),
 ('Tuned and Optimized Gradient Boosting Classifier',
  {'accuracy': 0.9391965255157437, 'no_features': 16}),
 ('Tuned Extreme Random Forest',
  {'accuracy': 0.9337676438653637, 'no_features': 58}),
 ('Random Forest Base Pipeline',
  {'accuracy': 0.9326818675352877, 'no_features': 58}),
 ('Tuned Multilayer Perceptor',
  {'accuracy': 0.9326818675352877, 'no_features': 58}),
 ('Tuned and Optimized Extreme Random Forest Classifier',
  {'accuracy': 0.9315960912052117, 'no_features': 9}),
 ('Multilayer Perceptor Classifier',
  {'accuracy': 0.9239956568946797, 'no_features': 58}),
 ('Tuned and Optimized Random Forest Classifier',
  {'accur

#### Endnotes

The journey of the Gradient Boosting Classifier through our experiments provides a clear lesson in the bias-variance tradeoff. We started with the base **Gradient Boosting Classifier**, which used all 58 features and achieved an accuracy of **92.4%**. Our first step to improve it was the **Tuned Gradient Boosting Classifier**. By optimizing its hyperparameters while still using all 58 features, we successfully boosted the accuracy to **94.6%**. This was a significant gain, achieved by reducing the model's bias without increasing variance, as all features were retained.

However, we wanted to see if we could simplify the model. The final version, the **Tuned and Optimized Gradient Boosting Classifier**, was a radical step towards reducing complexity. We dramatically cut the feature set from 58 down to just **16**, less than a third of the original. This simplification was meant to decrease variance and improve generalization. In practice, this came at a cost. The accuracy dropped from the tuned model's 94.6% down to **93.4%**.

So, was it worth it? The answer is a definitive **yes**. While we lost 1.2% in accuracy compared to the tuned version, the model became significantly less complex. It is now much faster to train and less prone to overfitting new data. We traded a small amount of bias for a large reduction in variance, creating a more robust model. This new, simpler model even still outperforms the base version by 1%.

Finally, when we look at the broader picture, the best-performing model overall is the **Extreme Random Forest Classifier** with 95.5% accuracy. The base Gradient Boosting (92.4%) was beaten by many other model types. However, our simplified Gradient Boosting (93.4%) is very competitive, matching the accuracy of the complex Tuned Neural Network and even outperforming the base Random Forest. It proves that a simpler, well-chosen model can be just as powerful as more complex alternatives.

### Extreme Gradient Boosting Classifier

In [86]:
from xgboost import XGBClassifier

XGB_MODEL_NAME = "Extreme Gradient Boosting"

In [87]:
numerical_features = df.drop(columns=["class"]).select_dtypes(include=np.number).columns
categorical_features = []

X = df.drop(columns=["class"])
y = df["class"]

X_train, X_test, y_train, y_test = train_test_split(  
	X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE  
)

#### Extreme Gradient Boosting Base Pipeline

In [90]:
extreme_gradient_boosting = XGBClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=4,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=1.0,
    random_state=42,
    eval_metric="logloss"
)

num_pipeline = Pipeline([  
	("imputer", SimpleImputer(strategy="median"))  
])  
  
cat_pipeline = Pipeline([  
	("imputer", SimpleImputer(strategy="most_frequent")),  
	("encoder", OneHotEncoder(handle_unknown="ignore"))  
])  
  
preprocessor = ColumnTransformer([  
	("num", num_pipeline, numerical_features),  
	("cat", cat_pipeline, categorical_features)  
])  
  
xgb_pipeline = Pipeline([  
	("preprocessing", preprocessor),  
	("xgb_classifier", extreme_gradient_boosting)
])

xgb_pipeline.fit(X_train, y_train)


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessing', ...), ('xgb_classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different trans

In [91]:
y_pred = xgb_pipeline.predict(X_test)
xgb_accuracy = accuracy_score(y_test, y_pred)
classifier_metrics[f"{XGB_MODEL_NAME} Classifier"] = {
    "accuracy": xgb_accuracy,
    "no_features": 58,
}
print_basic_metrics(model_name=f"{XGB_MODEL_NAME} Classifier", y_test=y_test, y_pred=y_pred)

EXTREME GRADIENT BOOSTING CLASSIFIER PERFORMANCE METRICS
Accuracy:                0.948
Precision (weighted):    0.947
Recall (weighted):       0.948
F1-Score (weighted):     0.948

Confusion Matrix:
-----------------
True Negatives:  515
False Positives: 20
False Negatives: 28
True Positives:  358

Classification Report:
              precision    recall  f1-score   support

           0       0.95      0.96      0.96       535
           1       0.95      0.93      0.94       386

    accuracy                           0.95       921
   macro avg       0.95      0.95      0.95       921
weighted avg       0.95      0.95      0.95       921



In [92]:
classifier_metrics

{'Random Forest Base Pipeline': {'accuracy': 0.9326818675352877,
  'no_features': 58},
 'Tuned Random Forest': {'accuracy': 0.9457111834961998, 'no_features': 58},
 'Tuned and Optimized Random Forest Classifier': {'accuracy': 0.9077090119435396,
  'no_features': 8},
 'Extreme Random Forest Classifier': {'accuracy': 0.9554831704668838,
  'no_features': 58},
 'Tuned Extreme Random Forest': {'accuracy': 0.9337676438653637,
  'no_features': 58},
 'Tuned and Optimized Extreme Random Forest Classifier': {'accuracy': 0.9315960912052117,
  'no_features': 9},
 'Multilayer Perceptor Classifier': {'accuracy': 0.9239956568946797,
  'no_features': 58},
 'Tuned Multilayer Perceptor': {'accuracy': 0.9326818675352877,
  'no_features': 58},
 'Gradient Boosting Classifier': {'accuracy': 0.9391965255157437,
  'no_features': 58},
 'Tuned Gradient Boosting Classifier': {'accuracy': 0.9457111834961998,
  'no_features': 58},
 'Tuned and Optimized Gradient Boosting Classifier': {'accuracy': 0.9391965255157437

#### Search for the Best Combination of Hyperparameters for Extreme Gradient Boosting

In [93]:
param_grid = {
    "xgb_classifier__n_estimators": [100, 200, 300],
    "xgb_classifier__max_depth": [3, 4, 5, 6],
    "xgb_classifier__learning_rate": [0.01, 0.05, 0.1, 0.2],
    "xgb_classifier__subsample": [0.6, 0.8, 1.0],
    "xgb_classifier__colsample_bytree": [0.6, 0.8, 1.0],
    "xgb_classifier__gamma": [0, 0.1, 0.3, 0.5],
    "xgb_classifier__reg_alpha": [0, 0.1, 1],
    "xgb_classifier__reg_lambda": [1, 1.5, 2]
}

random_xgb = RandomizedSearchCV(
    xgb_pipeline,
    param_distributions=param_grid,
    n_iter=20,
    cv=5,
    scoring="accuracy",
    n_jobs=-1,
    random_state=RANDOM_STATE
)

random_xgb.fit(X_train, y_train)

,"estimator estimator: estimator objectAn object of that type is instantiated for each grid point.This is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.","Pipeline(step...=None, ...))])"
,"param_distributions param_distributions: dict or list of dictsDictionary with parameters names (`str`) as keys and distributionsor lists of parameters to try. Distributions must provide a ``rvs``method for sampling (such as those from scipy.stats.distributions).If a list is given, it is sampled uniformly.If a list of dicts is given, first a dict is sampled uniformly, andthen a parameter is sampled using that dict as above.","{'xgb_classifier__colsample_bytree': [0.6, 0.8, ...], 'xgb_classifier__gamma': [0, 0.1, ...], 'xgb_classifier__learning_rate': [0.01, 0.05, ...], 'xgb_classifier__max_depth': [3, 4, ...], ...}"
,"n_iter n_iter: int, default=10Number of parameter settings that are sampled. n_iter tradesoff runtime vs quality of the solution.",20
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.If None, the estimator's score method is used.",'accuracy'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given the ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``RandomizedSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` f

In [94]:
best_xgb = random_xgb.best_estimator_  
classifer = best_xgb.named_steps["xgb_classifier"]

print("Best Parameters:", random_xgb.best_params_)
print("Best CV Score:", random_xgb.best_score_)

y_pred = best_xgb.predict(X_test)
tuned_xgb_accuracy = accuracy_score(y_test, y_pred)
print_basic_metrics(model_name=f"Tuned {XGB_MODEL_NAME} Classifier", y_test=y_test, y_pred=y_pred)

Best Parameters: {'xgb_classifier__subsample': 0.6, 'xgb_classifier__reg_lambda': 2, 'xgb_classifier__reg_alpha': 1, 'xgb_classifier__n_estimators': 100, 'xgb_classifier__max_depth': 6, 'xgb_classifier__learning_rate': 0.2, 'xgb_classifier__gamma': 0.5, 'xgb_classifier__colsample_bytree': 0.8}
Best CV Score: 0.9516304347826086
TUNED EXTREME GRADIENT BOOSTING CLASSIFIER PERFORMANCE METRICS
Accuracy:                0.950
Precision (weighted):    0.947
Recall (weighted):       0.950
F1-Score (weighted):     0.950

Confusion Matrix:
-----------------
True Negatives:  515
False Positives: 20
False Negatives: 26
True Positives:  360

Classification Report:
              precision    recall  f1-score   support

           0       0.95      0.96      0.96       535
           1       0.95      0.93      0.94       386

    accuracy                           0.95       921
   macro avg       0.95      0.95      0.95       921
weighted avg       0.95      0.95      0.95       921



In [95]:
classifier_metrics[f"Tuned {XGB_MODEL_NAME} Classifier"] = {
    "accuracy": tuned_xgb_accuracy,
    "no_features": 58
}

pprint.pprint(classifier_metrics)

{'Extreme Gradient Boosting Classifier': {'accuracy': 0.9478827361563518,
                                          'no_features': 58},
 'Extreme Random Forest Classifier': {'accuracy': 0.9554831704668838,
                                      'no_features': 58},
 'Gradient Boosting Classifier': {'accuracy': 0.9391965255157437,
                                  'no_features': 58},
 'Multilayer Perceptor Classifier': {'accuracy': 0.9239956568946797,
                                     'no_features': 58},
 'Random Forest Base Pipeline': {'accuracy': 0.9326818675352877,
                                 'no_features': 58},
 'Tuned Extreme Gradient Boosting Classifier': {'accuracy': 0.9500542888165038,
                                                'no_features': 58},
 'Tuned Extreme Random Forest': {'accuracy': 0.9337676438653637,
                                 'no_features': 58},
 'Tuned Gradient Boosting Classifier': {'accuracy': 0.9457111834961998,
                                  

#### Optimizing Extreme Gradient Boosting Classifier

In [96]:
importances = np.abs(classifer.feature_importances_)
feature_importance = pd.Series(importances, index=X_train.columns)
feature_importance = feature_importance.sort_values(ascending=False)  
print(feature_importance)

char_freq_%24                 0.154013
word_freq_remove              0.116680
char_freq_%21                 0.084148
word_freq_hp                  0.060839
word_freq_george              0.035350
word_freq_money               0.035106
word_freq_free                0.031456
word_freq_edu                 0.025546
word_freq_hpl                 0.023053
capital_run_length_longest    0.021570
word_freq_your                0.021429
word_freq_1999                0.020977
word_freq_650                 0.020847
word_freq_our                 0.020608
word_freq_meeting             0.020099
word_freq_85                  0.019321
capital_run_length_average    0.017542
word_freq_business            0.016148
word_freq_project             0.014688
word_freq_re                  0.014315
word_freq_credit              0.014271
word_freq_pm                  0.013406
word_freq_000                 0.012935
word_freq_internet            0.012437
capital_run_length_total      0.010985
word_freq_over           

In [97]:
important_features_scores = feature_importance[feature_importance > 0.02]
important_features = feature_importance[feature_importance > 0.02].index.to_list()
print(important_features_scores)
print(f"NUMBER OF FEATURES: {len(important_features)}")

char_freq_%24                 0.154013
word_freq_remove              0.116680
char_freq_%21                 0.084148
word_freq_hp                  0.060839
word_freq_george              0.035350
word_freq_money               0.035106
word_freq_free                0.031456
word_freq_edu                 0.025546
word_freq_hpl                 0.023053
capital_run_length_longest    0.021570
word_freq_your                0.021429
word_freq_1999                0.020977
word_freq_650                 0.020847
word_freq_our                 0.020608
word_freq_meeting             0.020099
dtype: float32
NUMBER OF FEATURES: 15


In [98]:
X_train_optimized = X_train[important_features]
X_test_optimized = X_test[important_features]

numerical_features = (
    df
    .drop(columns=["class"])
    .loc[:, important_features]
    .select_dtypes(include=np.number)
    .columns
)

In [99]:
# Best Parameters: 
# {'xgb_classifier__subsample': 0.6,
# 'xgb_classifier__reg_lambda': 2, 
# 'xgb_classifier__reg_alpha': 1, 
# 'xgb_classifier__n_estimators': 100, 
# 'xgb_classifier__max_depth': 6, 
# 'xgb_classifier__learning_rate': 0.2, 
# 'xgb_classifier__gamma': 0.5, 
# 'xgb_classifier__colsample_bytree': 0.8}


In [100]:
extreme_gradient_boosting = XGBClassifier(
    n_estimators=100,
    learning_rate=0.2,
    max_depth=6,
    subsample=0.6,
    colsample_bytree=0.8,
    reg_alpha=2.0,
    reg_lambda=1.0,
    random_state=RANDOM_STATE,
    eval_metric="logloss",
    gamma=0.5
)

num_pipeline = Pipeline([  
	("imputer", SimpleImputer(strategy="median"))  
])  
  
cat_pipeline = Pipeline([  
	("imputer", SimpleImputer(strategy="most_frequent")),  
	("encoder", OneHotEncoder(handle_unknown="ignore"))  
])  
  
preprocessor = ColumnTransformer([  
	("num", num_pipeline, numerical_features),  
	("cat", cat_pipeline, categorical_features)  
])  
  
xgb_pipeline = Pipeline([  
	("preprocessing", preprocessor),  
	("xgb_classifier", extreme_gradient_boosting)
])

xgb_pipeline.fit(X_train_optimized, y_train)


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessing', ...), ('xgb_classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different trans

In [101]:
y_pred = xgb_pipeline.predict(X_test)
xgb_accuracy = accuracy_score(y_test, y_pred)
print_basic_metrics(
    model_name=f"Tuned and Optimized {XGB_MODEL_NAME} Classifier", 
    y_test=y_test, 
    y_pred=y_pred
)

TUNED AND OPTIMIZED EXTREME GRADIENT BOOSTING CLASSIFIER PERFORMANCE METRICS
Accuracy:                0.937
Precision (weighted):    0.936
Recall (weighted):       0.937
F1-Score (weighted):     0.937

Confusion Matrix:
-----------------
True Negatives:  511
False Positives: 24
False Negatives: 34
True Positives:  352

Classification Report:
              precision    recall  f1-score   support

           0       0.94      0.96      0.95       535
           1       0.94      0.91      0.92       386

    accuracy                           0.94       921
   macro avg       0.94      0.93      0.94       921
weighted avg       0.94      0.94      0.94       921



In [102]:
classifier_metrics[f"Tuned and Optimized {XGB_MODEL_NAME} Classifier"] = {
    "accuracy": xgb_accuracy,
    "no_features": len(important_features_scores)
}

In [103]:
pprint.pprint(sorted(classifier_metrics.items(), key=lambda stats: stats[1]["accuracy"], reverse=True))

[('Extreme Random Forest Classifier',
  {'accuracy': 0.9554831704668838, 'no_features': 58}),
 ('Tuned Extreme Gradient Boosting Classifier',
  {'accuracy': 0.9500542888165038, 'no_features': 58}),
 ('Extreme Gradient Boosting Classifier',
  {'accuracy': 0.9478827361563518, 'no_features': 58}),
 ('Tuned Random Forest', {'accuracy': 0.9457111834961998, 'no_features': 58}),
 ('Tuned Gradient Boosting Classifier',
  {'accuracy': 0.9457111834961998, 'no_features': 58}),
 ('Gradient Boosting Classifier',
  {'accuracy': 0.9391965255157437, 'no_features': 58}),
 ('Tuned and Optimized Gradient Boosting Classifier',
  {'accuracy': 0.9391965255157437, 'no_features': 16}),
 ('Tuned and Optimized Extreme Gradient Boosting Classifier',
  {'accuracy': 0.9370249728555917, 'no_features': 15}),
 ('Tuned Extreme Random Forest',
  {'accuracy': 0.9337676438653637, 'no_features': 58}),
 ('Random Forest Base Pipeline',
  {'accuracy': 0.9326818675352877, 'no_features': 58}),
 ('Tuned Multilayer Perceptor',
 

#### Endnotes

The evolution of the Extreme Gradient Boosting (XGBoost) model across our experiments is a masterclass in the bias-variance tradeoff, showcasing that "more" is not always "better." We began our journey with the baseline **Extreme Gradient Boosting Classifier**, which utilized all 58 features and delivered a solid accuracy of **94.8%**. This was a strong starting point, but we believed we could push its performance further.

Our first optimization step was the **Tuned Extreme Gradient Boosting Classifier**. By carefully adjusting its hyperparameters while keeping all 58 features, we successfully reduced the model's bias, boosting accuracy to an impressive **95.0%**. This was a clear win, demonstrating that fine-tuning could extract more predictive power from the existing data without increasing variance.

Encouraged by this success, we decided to explore the other side of the tradeoff: reducing variance by simplifying the model. The final iteration, the **Tuned and Optimized Extreme Gradient Boosting Classifier**, underwent a radical feature reduction, slashing the feature set from 58 down to just **15**. This simplification came with a predictable cost: accuracy dropped to **93.7%**, a 1.3% decrease from the tuned version.

So, was this tradeoff worth it? Absolutely. We sacrificed a small margin of accuracy for a massive reduction in model complexity. The new model is leaner, trains faster, and is far less susceptible to overfitting, making it a more reliable and generalizable solution for new data. This principle of parsimony is invaluable. In fact, our simplified XGBoost still outperforms the baseline XGBoost from which we started, proving that we have built a more efficient model.

When comparing it to the broader landscape of models, the results are telling. The ultimate champion remains the **Extreme Random Forest Classifier** with a stellar 95.5% accuracy. However, our tuned XGBoost (95.0%) sits right behind it, outperforming the Tuned Random Forest and Tuned Gradient Boosting (both at 94.6%). Even our simplified XGBoost (93.7%) remains fiercely competitive, matching the Tuned Extreme Random Forest and surpassing both the base Random Forest and the Tuned Neural Network. This solidifies XGBoost as one of the most powerful and flexible tools in our arsenal, capable of delivering top-tier performance with elegance and efficiency.

### Light Gradient Boosting Classifier 

In [104]:
from lightgbm import LGBMClassifier

LGBM_MODEL_NAME = "Light Gradient Boosting"

In [105]:
numerical_features = df.drop(columns=["class"]).select_dtypes(include=np.number).columns
categorical_features = []

X = df.drop(columns=["class"])
y = df["class"]

X_train, X_test, y_train, y_test = train_test_split(  
	X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE  
)

#### Light Gradient Boosting Classifier Base Pipeline

In [106]:
lgbm = LGBMClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=-1,
    num_leaves=31,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=RANDOM_STATE
)


num_pipeline = Pipeline([  
	("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())  
])  
  
cat_pipeline = Pipeline([  
	("imputer", SimpleImputer(strategy="most_frequent")),  
	("encoder", OneHotEncoder(handle_unknown="ignore"))  
])  
  
preprocessor = ColumnTransformer([  
	("num", num_pipeline, numerical_features),  
	("cat", cat_pipeline, categorical_features)  
])  
  
lgbm_pipeline = Pipeline([  
	("preprocessing", preprocessor),  
	("lgbm_classifier", lgbm)
])

lgbm_pipeline.fit(X_train, y_train)

[LightGBM] [Info] Number of positive: 1427, number of negative: 2253
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000892 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 6996
[LightGBM] [Info] Number of data points in the train set: 3680, number of used features: 57
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.387772 -> initscore=-0.456688
[LightGBM] [Info] Start training from score -0.456688


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessing', ...), ('lgbm_classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different tran

In [107]:
y_pred = lgbm_pipeline.predict(X_test)
lgbm_accuracy = accuracy_score(y_test, y_pred)
classifier_metrics[f"{LGBM_MODEL_NAME} Classifier"] = {
    "accuracy": lgbm_accuracy,
    "no_features": 58,
}
print_basic_metrics(model_name=f"{LGBM_MODEL_NAME} Classifier", y_test=y_test, y_pred=y_pred)

LIGHT GRADIENT BOOSTING CLASSIFIER PERFORMANCE METRICS
Accuracy:                0.950
Precision (weighted):    0.947
Recall (weighted):       0.950
F1-Score (weighted):     0.950

Confusion Matrix:
-----------------
True Negatives:  515
False Positives: 20
False Negatives: 26
True Positives:  360

Classification Report:
              precision    recall  f1-score   support

           0       0.95      0.96      0.96       535
           1       0.95      0.93      0.94       386

    accuracy                           0.95       921
   macro avg       0.95      0.95      0.95       921
weighted avg       0.95      0.95      0.95       921



In [108]:
classifier_metrics

{'Random Forest Base Pipeline': {'accuracy': 0.9326818675352877,
  'no_features': 58},
 'Tuned Random Forest': {'accuracy': 0.9457111834961998, 'no_features': 58},
 'Tuned and Optimized Random Forest Classifier': {'accuracy': 0.9077090119435396,
  'no_features': 8},
 'Extreme Random Forest Classifier': {'accuracy': 0.9554831704668838,
  'no_features': 58},
 'Tuned Extreme Random Forest': {'accuracy': 0.9337676438653637,
  'no_features': 58},
 'Tuned and Optimized Extreme Random Forest Classifier': {'accuracy': 0.9315960912052117,
  'no_features': 9},
 'Multilayer Perceptor Classifier': {'accuracy': 0.9239956568946797,
  'no_features': 58},
 'Tuned Multilayer Perceptor': {'accuracy': 0.9326818675352877,
  'no_features': 58},
 'Gradient Boosting Classifier': {'accuracy': 0.9391965255157437,
  'no_features': 58},
 'Tuned Gradient Boosting Classifier': {'accuracy': 0.9457111834961998,
  'no_features': 58},
 'Tuned and Optimized Gradient Boosting Classifier': {'accuracy': 0.9391965255157437

#### Search for the Best Combination of Hyperparameters for Light Gradient Boosting Classifier

In [ ]:
param_grid = {
    "lgbm_classifier__n_estimators": [100, 200, 300],
    "lgbm_classifier__num_leaves": [31, 50, 70, 100],
    "lgbm_classifier__max_depth": [-1, 3, 5, 7],
    "lgbm_classifier__learning_rate": [0.01, 0.05, 0.1, 0.2],
    "lgbm_classifier__subsample": [0.6, 0.8, 1.0],
    "lgbm_classifier__colsample_bytree": [0.6, 0.8, 1.0],
    "lgbm_classifier__reg_alpha": [0, 0.1, 1],
    "lgbm_classifier__reg_lambda": [0, 0.1, 1],
    "lgbm_classifier__min_child_samples": [20, 50, 100]
}

random_lgbm = RandomizedSearchCV(
    lgbm_pipeline,
    param_distributions=param_grid,
    n_iter=20,
    cv=5,
    scoring="accuracy",
    n_jobs=-1,
    random_state=RANDOM_STATE
)

random_lgbm.fit(X_train, y_train)

[LightGBM] [Info] Number of positive: 1427, number of negative: 2253
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.001283 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 6981
[LightGBM] [Info] Number of data points in the train set: 3680, number of used features: 56
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.387772 -> initscore=-0.456688
[LightGBM] [Info] Start training from score -0.456688
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, 

,"estimator estimator: estimator objectAn object of that type is instantiated for each grid point.This is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...sample=0.8))])
,"param_distributions param_distributions: dict or list of dictsDictionary with parameters names (`str`) as keys and distributionsor lists of parameters to try. Distributions must provide a ``rvs``method for sampling (such as those from scipy.stats.distributions).If a list is given, it is sampled uniformly.If a list of dicts is given, first a dict is sampled uniformly, andthen a parameter is sampled using that dict as above.","{'lgbm_classifier__colsample_bytree': [0.6, 0.8, ...], 'lgbm_classifier__learning_rate': [0.01, 0.05, ...], 'lgbm_classifier__max_depth': [-1, 3, ...], 'lgbm_classifier__min_child_samples': [20, 50, ...], ...}"
,"n_iter n_iter: int, default=10Number of parameter settings that are sampled. n_iter tradesoff runtime vs quality of the solution.",20
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.If None, the estimator's score method is used.",'accuracy'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given the ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``RandomizedSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:

In [ ]:
best_lgbm = random_lgbm.best_estimator_  
classifer = best_lgbm.named_steps["lgbm_classifier"]

print("Best Parameters:", random_lgbm.best_params_)
print("Best CV Score:", random_lgbm.best_score_)

y_pred = best_lgbm.predict(X_test)
tuned_lgbm_accuracy = accuracy_score(y_test, y_pred)
print_basic_metrics(model_name=f"Tuned {LGBM_MODEL_NAME} Classifier", y_test=y_test, y_pred=y_pred)

Best Parameters: {'lgbm_classifier__subsample': 1.0, 'lgbm_classifier__reg_lambda': 0, 'lgbm_classifier__reg_alpha': 0, 'lgbm_classifier__num_leaves': 100, 'lgbm_classifier__n_estimators': 300, 'lgbm_classifier__min_child_samples': 50, 'lgbm_classifier__max_depth': 5, 'lgbm_classifier__learning_rate': 0.1, 'lgbm_classifier__colsample_bytree': 0.8}
Best CV Score: 0.9567934782608696
TUNED LIGHT GRADIENT BOOSTING CLASSIFIER PERFORMANCE METRICS
Accuracy:                0.950
Precision (weighted):    0.955
Recall (weighted):       0.950
F1-Score (weighted):     0.950

Confusion Matrix:
-----------------
True Negatives:  518
False Positives: 17
False Negatives: 29
True Positives:  357

Classification Report:
              precision    recall  f1-score   support

           0       0.95      0.97      0.96       535
           1       0.95      0.92      0.94       386

    accuracy                           0.95       921
   macro avg       0.95      0.95      0.95       921
weighted avg    

In [111]:
classifier_metrics[f"Tuned {LGBM_MODEL_NAME} Classifier"] = {
    "accuracy": tuned_lgbm_accuracy,
    "no_features": 58
}

pprint.pprint(classifier_metrics)

{'Extreme Gradient Boosting Classifier': {'accuracy': 0.9478827361563518,
                                          'no_features': 58},
 'Extreme Random Forest Classifier': {'accuracy': 0.9554831704668838,
                                      'no_features': 58},
 'Gradient Boosting Classifier': {'accuracy': 0.9391965255157437,
                                  'no_features': 58},
 'Light Gradient Boosting Classifier': {'accuracy': 0.9500542888165038,
                                        'no_features': 58},
 'Multilayer Perceptor Classifier': {'accuracy': 0.9239956568946797,
                                     'no_features': 58},
 'Random Forest Base Pipeline': {'accuracy': 0.9326818675352877,
                                 'no_features': 58},
 'Tuned Extreme Gradient Boosting Classifier': {'accuracy': 0.9500542888165038,
                                                'no_features': 58},
 'Tuned Extreme Random Forest': {'accuracy': 0.9337676438653637,
                           

#### Optimizing LGBM

In [ ]:
importances = np.abs(classifer.feature_importances_)
feature_importance = pd.Series(importances, index=X_train.columns)
feature_importance = feature_importance.sort_values(ascending=False)  
print(feature_importance)

capital_run_length_total      306
word_freq_you                 288
capital_run_length_average    256
char_freq_%21                 254
capital_run_length_longest    206
word_freq_your                180
word_freq_will                158
word_freq_free                154
char_freq_%28                 143
word_freq_our                 100
char_freq_%24                  97
word_freq_remove               95
word_freq_all                  95
word_freq_re                   91
word_freq_hp                   89
word_freq_edu                  88
word_freq_mail                 85
word_freq_over                 60
word_freq_email                58
word_freq_business             55
word_freq_1999                 54
word_freq_george               52
char_freq_%3B                  50
word_freq_meeting              47
word_freq_internet             45
word_freq_000                  43
word_freq_money                43
word_freq_pm                   41
word_freq_receive              38
word_freq_proj

In [113]:
important_features_scores = feature_importance[feature_importance > 90]
important_features = feature_importance[feature_importance > 90].index.to_list()
print(important_features_scores)
print(f"NUMBER OF FEATURES: {len(important_features)}")

capital_run_length_total      306
word_freq_you                 288
capital_run_length_average    256
char_freq_%21                 254
capital_run_length_longest    206
word_freq_your                180
word_freq_will                158
word_freq_free                154
char_freq_%28                 143
word_freq_our                 100
char_freq_%24                  97
word_freq_remove               95
word_freq_all                  95
word_freq_re                   91
dtype: int32
NUMBER OF FEATURES: 14


In [114]:
X_train_optimized = X_train[important_features]
X_test_optimized = X_test[important_features]

numerical_features = (
    df
    .drop(columns=["class"])
    .loc[:, important_features]
    .select_dtypes(include=np.number)
    .columns
)

In [115]:
# Best Parameters: 
# {'lgbm_classifier__subsample': 1.0, 
# 'lgbm_classifier__reg_lambda': 0, 
# 'lgbm_classifier__reg_alpha': 0, 
# 'lgbm_classifier__num_leaves': 100, 
# 'lgbm_classifier__n_estimators': 300, 
# 'lgbm_classifier__min_child_samples': 50, 
# 'lgbm_classifier__max_depth': 5, 
# 'lgbm_classifier__learning_rate': 0.1, 
# 'lgbm_classifier__colsample_bytree': 0.8}


In [116]:
lgbm = LGBMClassifier(
    n_estimators=300,
    learning_rate=0.1,
    max_depth=5,
    num_leaves=100,
    subsample=1.0,
    colsample_bytree=0.8,
    reg_alpha=0,
    reg_lambda=0,
    min_child_samples=50,
    random_state=RANDOM_STATE,
    eval_metric="logloss"
)

num_pipeline = Pipeline([  
	("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())  
])  
  
cat_pipeline = Pipeline([  
	("imputer", SimpleImputer(strategy="most_frequent")),  
	("encoder", OneHotEncoder(handle_unknown="ignore"))  
])  
  
preprocessor = ColumnTransformer([  
	("num", num_pipeline, numerical_features),  
	("cat", cat_pipeline, categorical_features)  
])  
  
lgbm_pipeline = Pipeline([  
	("preprocessing", preprocessor),  
	("lgbm_classifier", lgbm)
])

lgbm_pipeline.fit(X_train, y_train)

[LightGBM] [Warning] Unknown parameter: eval_metric
[LightGBM] [Warning] Unknown parameter: eval_metric
[LightGBM] [Info] Number of positive: 1427, number of negative: 2253
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000328 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 3032
[LightGBM] [Info] Number of data points in the train set: 3680, number of used features: 14
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.387772 -> initscore=-0.456688
[LightGBM] [Info] Start training from score -0.456688
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No furthe

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessing', ...), ('lgbm_classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different tran

In [117]:
y_pred = lgbm_pipeline.predict(X_test)
lgbm_accuracy = accuracy_score(y_test, y_pred)
print_basic_metrics(
    model_name=f"Tuned and Optimized {LGBM_MODEL_NAME} Classifier", 
    y_test=y_test, 
    y_pred=y_pred
)

[LightGBM] [Warning] Unknown parameter: eval_metric
TUNED AND OPTIMIZED LIGHT GRADIENT BOOSTING CLASSIFIER PERFORMANCE METRICS
Accuracy:                0.921
Precision (weighted):    0.929
Recall (weighted):       0.921
F1-Score (weighted):     0.920

Confusion Matrix:
-----------------
True Negatives:  509
False Positives: 26
False Negatives: 47
True Positives:  339

Classification Report:
              precision    recall  f1-score   support

           0       0.92      0.95      0.93       535
           1       0.93      0.88      0.90       386

    accuracy                           0.92       921
   macro avg       0.92      0.91      0.92       921
weighted avg       0.92      0.92      0.92       921



In [118]:
classifier_metrics[f"Tuned and Optimized {LGBM_MODEL_NAME} Classifier"] = {
    "accuracy": lgbm_accuracy,
    "no_features": len(important_features_scores)
}

In [119]:
pprint.pprint(sorted(classifier_metrics.items(), key=lambda stats: stats[1]["accuracy"], reverse=True))

[('Extreme Random Forest Classifier',
  {'accuracy': 0.9554831704668838, 'no_features': 58}),
 ('Tuned Extreme Gradient Boosting Classifier',
  {'accuracy': 0.9500542888165038, 'no_features': 58}),
 ('Light Gradient Boosting Classifier',
  {'accuracy': 0.9500542888165038, 'no_features': 58}),
 ('Tuned Light Gradient Boosting Classifier',
  {'accuracy': 0.9500542888165038, 'no_features': 58}),
 ('Extreme Gradient Boosting Classifier',
  {'accuracy': 0.9478827361563518, 'no_features': 58}),
 ('Tuned Random Forest', {'accuracy': 0.9457111834961998, 'no_features': 58}),
 ('Tuned Gradient Boosting Classifier',
  {'accuracy': 0.9457111834961998, 'no_features': 58}),
 ('Gradient Boosting Classifier',
  {'accuracy': 0.9391965255157437, 'no_features': 58}),
 ('Tuned and Optimized Gradient Boosting Classifier',
  {'accuracy': 0.9391965255157437, 'no_features': 16}),
 ('Tuned and Optimized Extreme Gradient Boosting Classifier',
  {'accuracy': 0.9370249728555917, 'no_features': 15}),
 ('Tuned Extr

#### Endnotes

The Light Gradient Boosting Machine (LightGBM) tells a compelling story about the delicate balance between performance and efficiency. Our journey began with the base **Light Gradient Boosting Classifier**, which used all 58 features and achieved a strong accuracy of **95.0%**. This was an exceptional starting point, placing it among the top performers right out of the gate.

Naturally, we wondered if we could squeeze even more performance through hyperparameter tuning. The **Tuned Light Gradient Boosting Classifier** retained all 58 features, but despite our best efforts, it plateaued at the exact same accuracy of **95.0%**. This was an important insight—the model had already reached its performance ceiling with the full feature set, and tuning alone couldn't push it further.

This plateau prompted us to take a different approach. Instead of chasing marginal gains through hyperparameters, we decided to simplify the model drastically. The final iteration, the **Tuned and Optimized Light Gradient Boosting Classifier**, underwent a massive feature reduction from 58 down to just **14 features**. This was a bold move designed to reduce variance and improve generalization by eliminating noisy or redundant predictors.

However, this simplification came at a significant cost. The accuracy dropped from 95.0% to **92.1%**, a loss of nearly 3 percentage points. Was this tradeoff worth it? In this case, the answer is arguably **no**. While we achieved a much simpler and faster model, the performance penalty was too severe. The 3% accuracy drop represents a meaningful degradation in predictive power that may not be justified by the gains in efficiency alone. Unlike our previous experiences with XGBoost where simplification cost only 1.3%, LightGBM proved more sensitive to feature reduction.

When we compare LightGBM to the broader model landscape, the results are revealing. The base LightGBM (95.0%) stands as a joint second-best performer, tying with the Tuned Extreme Gradient Boosting and trailing only the Extreme Random Forest (95.5%). It comfortably outperforms the Tuned Random Forest (94.6%) and the base Gradient Boosting (93.9%). However, the simplified LightGBM (92.1%) falls behind even the base Random Forest (93.3%) and the Tuned Multilayer Perceptron (93.3%). This stark contrast teaches us a valuable lesson: sometimes, the pursuit of simplicity can go too far, and LightGBM's power lies in leveraging its full feature set to deliver near-state-of-the-art performance.

### CatBoost Classifier

In [120]:
from catboost import CatBoostClassifier

CATBOOST_MODEL_NAME = "CatBoost"

In [121]:
numerical_features = df.drop(columns=["class"]).select_dtypes(include=np.number).columns
categorical_features = []

X = df.drop(columns=["class"])
y = df["class"]

X_train, X_test, y_train, y_test = train_test_split(  
	X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE  
)

#### CatBoost Classifier Base Pipeline

In [122]:
catboost = CatBoostClassifier(
    iterations=300,
    learning_rate=0.05,
    depth=6,
    l2_leaf_reg=3.0,
    random_state=42,
    verbose=False
)


num_pipeline = Pipeline([  
	("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())  
])  
  
cat_pipeline = Pipeline([  
	("imputer", SimpleImputer(strategy="most_frequent")),  
	("encoder", OneHotEncoder(handle_unknown="ignore"))  
])  
  
preprocessor = ColumnTransformer([  
	("num", num_pipeline, numerical_features),  
	("cat", cat_pipeline, categorical_features)  
])  
  
catboost_pipeline = Pipeline([  
	("preprocessing", preprocessor),  
	("catboost_classifier", catboost)
])

catboost_pipeline.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessing', ...), ('catboost_classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different 

In [123]:
y_pred = catboost_pipeline.predict(X_test)
catboost_accuracy = accuracy_score(y_test, y_pred)
classifier_metrics[f"{CATBOOST_MODEL_NAME} Classifier"] = {
    "accuracy": catboost_accuracy,
    "no_features": 58,
}
print_basic_metrics(model_name=f"{CATBOOST_MODEL_NAME} Classifier", y_test=y_test, y_pred=y_pred)

CATBOOST CLASSIFIER PERFORMANCE METRICS
Accuracy:                0.942
Precision (weighted):    0.932
Recall (weighted):       0.942
F1-Score (weighted):     0.942

Confusion Matrix:
-----------------
True Negatives:  509
False Positives: 26
False Negatives: 27
True Positives:  359

Classification Report:
              precision    recall  f1-score   support

           0       0.95      0.95      0.95       535
           1       0.93      0.93      0.93       386

    accuracy                           0.94       921
   macro avg       0.94      0.94      0.94       921
weighted avg       0.94      0.94      0.94       921



In [130]:
classifier_metrics

{'Random Forest Base Pipeline': {'accuracy': 0.9326818675352877,
  'no_features': 58},
 'Tuned Random Forest': {'accuracy': 0.9457111834961998, 'no_features': 58},
 'Tuned and Optimized Random Forest Classifier': {'accuracy': 0.9077090119435396,
  'no_features': 8},
 'Extreme Random Forest Classifier': {'accuracy': 0.9554831704668838,
  'no_features': 58},
 'Tuned Extreme Random Forest': {'accuracy': 0.9337676438653637,
  'no_features': 58},
 'Tuned and Optimized Extreme Random Forest Classifier': {'accuracy': 0.9315960912052117,
  'no_features': 9},
 'Multilayer Perceptor Classifier': {'accuracy': 0.9239956568946797,
  'no_features': 58},
 'Tuned Multilayer Perceptor': {'accuracy': 0.9326818675352877,
  'no_features': 58},
 'Gradient Boosting Classifier': {'accuracy': 0.9391965255157437,
  'no_features': 58},
 'Tuned Gradient Boosting Classifier': {'accuracy': 0.9457111834961998,
  'no_features': 58},
 'Tuned and Optimized Gradient Boosting Classifier': {'accuracy': 0.9391965255157437

#### Search for the Best Combination of Hyperparameters for CatBoost Classifier

In [127]:
param_grid = {
    "catboost_classifier__iterations": [100, 200, 300],  
    "catboost_classifier__depth": [3, 5, 7, 9],
    "catboost_classifier__learning_rate": [0.01, 0.05, 0.1, 0.2],
    "catboost_classifier__subsample": [0.6, 0.8, 1.0],
    "catboost_classifier__colsample_bylevel": [0.6, 0.8, 1.0],
    "catboost_classifier__l2_leaf_reg": [0, 0.1, 1],  
    "catboost_classifier__min_child_samples": [20, 50, 100],
}

random_catboost = RandomizedSearchCV(
    catboost_pipeline,
    param_distributions=param_grid,
    n_iter=20,
    cv=5,
    scoring="accuracy",
    n_jobs=-1,
    random_state=RANDOM_STATE
)

random_catboost.fit(X_train, y_train)

,"estimator estimator: estimator objectAn object of that type is instantiated for each grid point.This is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...bose=False))])
,"param_distributions param_distributions: dict or list of dictsDictionary with parameters names (`str`) as keys and distributionsor lists of parameters to try. Distributions must provide a ``rvs``method for sampling (such as those from scipy.stats.distributions).If a list is given, it is sampled uniformly.If a list of dicts is given, first a dict is sampled uniformly, andthen a parameter is sampled using that dict as above.","{'catboost_classifier__colsample_bylevel': [0.6, 0.8, ...], 'catboost_classifier__depth': [3, 5, ...], 'catboost_classifier__iterations': [100, 200, ...], 'catboost_classifier__l2_leaf_reg': [0, 0.1, ...], ...}"
,"n_iter n_iter: int, default=10Number of parameter settings that are sampled. n_iter tradesoff runtime vs quality of the solution.",20
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.If None, the estimator's score method is used.",'accuracy'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given the ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``RandomizedSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref

In [129]:
best_catboost = random_catboost.best_estimator_  
classifer = best_catboost.named_steps["catboost_classifier"]

print("Best Parameters:", random_catboost.best_params_)
print("Best CV Score:", random_catboost.best_score_)

y_pred = best_catboost.predict(X_test)
tuned_catboost_accuracy = accuracy_score(y_test, y_pred)
print_basic_metrics(model_name=f"Tuned {CATBOOST_MODEL_NAME} Classifier", y_test=y_test, y_pred=y_pred)

Best Parameters: {'catboost_classifier__subsample': 0.6, 'catboost_classifier__min_child_samples': 20, 'catboost_classifier__learning_rate': 0.2, 'catboost_classifier__l2_leaf_reg': 1, 'catboost_classifier__iterations': 200, 'catboost_classifier__depth': 7, 'catboost_classifier__colsample_bylevel': 0.8}
Best CV Score: 0.9592391304347826
TUNED CATBOOST CLASSIFIER PERFORMANCE METRICS
Accuracy:                0.946
Precision (weighted):    0.944
Recall (weighted):       0.946
F1-Score (weighted):     0.946

Confusion Matrix:
-----------------
True Negatives:  514
False Positives: 21
False Negatives: 29
True Positives:  357

Classification Report:
              precision    recall  f1-score   support

           0       0.95      0.96      0.95       535
           1       0.94      0.92      0.93       386

    accuracy                           0.95       921
   macro avg       0.95      0.94      0.94       921
weighted avg       0.95      0.95      0.95       921



In [131]:
classifier_metrics[f"Tuned {CATBOOST_MODEL_NAME} Classifier"] = {
    "accuracy": tuned_catboost_accuracy,
    "no_features": 58
}

pprint.pprint(classifier_metrics)

{'CatBoost Classifier': {'accuracy': 0.9424538545059717, 'no_features': 58},
 'Extreme Gradient Boosting Classifier': {'accuracy': 0.9478827361563518,
                                          'no_features': 58},
 'Extreme Random Forest Classifier': {'accuracy': 0.9554831704668838,
                                      'no_features': 58},
 'Gradient Boosting Classifier': {'accuracy': 0.9391965255157437,
                                  'no_features': 58},
 'Light Gradient Boosting Classifier': {'accuracy': 0.9500542888165038,
                                        'no_features': 58},
 'Multilayer Perceptor Classifier': {'accuracy': 0.9239956568946797,
                                     'no_features': 58},
 'Random Forest Base Pipeline': {'accuracy': 0.9326818675352877,
                                 'no_features': 58},
 'Tuned CatBoost Classifier': {'accuracy': 0.9457111834961998,
                               'no_features': 58},
 'Tuned Extreme Gradient Boosting Classifier': {'

#### Optimizing CatBoost Classifier

In [ ]:
importances = np.abs(classifer.feature_importances_)
feature_importance = pd.Series(importances, index=X_train.columns)
feature_importance = feature_importance.sort_values(ascending=False)  
print(feature_importance)

word_freq_george              10.336616
word_freq_hp                   7.270365
capital_run_length_average     6.268333
capital_run_length_longest     5.934324
char_freq_%21                  5.532921
char_freq_%24                  4.427405
word_freq_remove               4.392892
capital_run_length_total       4.247730
word_freq_edu                  3.938186
word_freq_free                 3.164758
word_freq_you                  3.092677
word_freq_our                  3.052979
word_freq_your                 2.964509
word_freq_will                 2.758455
word_freq_meeting              2.640583
char_freq_%28                  2.545807
word_freq_re                   2.445261
word_freq_business             1.811086
word_freq_1999                 1.725523
word_freq_email                1.381197
word_freq_money                1.374051
word_freq_85                   1.264360
word_freq_000                  1.224947
word_freq_technology           1.187199
word_freq_mail                 0.945775


In [135]:
important_features_scores = feature_importance[feature_importance > 3]
important_features = feature_importance[feature_importance > 3].index.to_list()
print(important_features_scores)
print(f"NUMBER OF FEATURES: {len(important_features)}")

word_freq_george              10.336616
word_freq_hp                   7.270365
capital_run_length_average     6.268333
capital_run_length_longest     5.934324
char_freq_%21                  5.532921
char_freq_%24                  4.427405
word_freq_remove               4.392892
capital_run_length_total       4.247730
word_freq_edu                  3.938186
word_freq_free                 3.164758
word_freq_you                  3.092677
word_freq_our                  3.052979
dtype: float64
NUMBER OF FEATURES: 12


In [137]:
X_train_optimized = X_train[important_features]
X_test_optimized = X_test[important_features]

numerical_features = (
    df
    .drop(columns=["class"])
    .loc[:, important_features]
    .select_dtypes(include=np.number)
    .columns
)

categorical_features = (
	df
	.drop(columns=["class"])
	.select_dtypes(include=["string", "object"])
	.columns
)

In [138]:
# Best Parameters: 
# {'catboost_classifier__subsample': 0.6, 
# 'catboost_classifier__min_child_samples': 20, 
# 'catboost_classifier__learning_rate': 0.2, 
# 'catboost_classifier__l2_leaf_reg': 1, 
# 'catboost_classifier__iterations': 200, 
# 'catboost_classifier__depth': 7, 
# 'catboost_classifier__colsample_bylevel': 0.8}

In [139]:
catboost = CatBoostClassifier(
    iterations=200,
    depth=7,
    learning_rate=0.2,
    subsample=0.6,
    colsample_bylevel=0.8,
    l2_leaf_reg=1,
    min_child_samples=20,
    random_seed=RANDOM_STATE,
    verbose=100,  
)

num_pipeline = Pipeline([  
	("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())  
])  
  
cat_pipeline = Pipeline([  
	("imputer", SimpleImputer(strategy="most_frequent")),  
	("encoder", OneHotEncoder(handle_unknown="ignore"))  
])  
  
preprocessor = ColumnTransformer([  
	("num", num_pipeline, numerical_features),  
	("cat", cat_pipeline, categorical_features)  
])  
  
catboost_pipeline = Pipeline([  
	("preprocessing", preprocessor),  
	("catboost_classifier", catboost)
])

catboost_pipeline.fit(X_train, y_train)

0:	learn: 0.4736099	total: 4.17ms	remaining: 829ms
100:	learn: 0.0475837	total: 472ms	remaining: 463ms
199:	learn: 0.0188322	total: 889ms	remaining: 0us


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessing', ...), ('catboost_classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different 

In [140]:
y_pred = catboost_pipeline.predict(X_test)
catboost_accuracy = accuracy_score(y_test, y_pred)
print_basic_metrics(
    model_name=f"Tuned and Optimized {CATBOOST_MODEL_NAME} Classifier", 
    y_test=y_test, 
    y_pred=y_pred
)

TUNED AND OPTIMIZED CATBOOST CLASSIFIER PERFORMANCE METRICS
Accuracy:                0.936
Precision (weighted):    0.936
Recall (weighted):       0.936
F1-Score (weighted):     0.936

Confusion Matrix:
-----------------
True Negatives:  511
False Positives: 24
False Negatives: 35
True Positives:  351

Classification Report:
              precision    recall  f1-score   support

           0       0.94      0.96      0.95       535
           1       0.94      0.91      0.92       386

    accuracy                           0.94       921
   macro avg       0.94      0.93      0.93       921
weighted avg       0.94      0.94      0.94       921



In [141]:
classifier_metrics[f"Tuned and Optimized {CATBOOST_MODEL_NAME} Classifier"] = {
    "accuracy": catboost_accuracy,
    "no_features": len(important_features_scores)
}

In [142]:
pprint.pprint(sorted(classifier_metrics.items(), key=lambda stats: stats[1]["accuracy"], reverse=True))

[('Extreme Random Forest Classifier',
  {'accuracy': 0.9554831704668838, 'no_features': 58}),
 ('Tuned Extreme Gradient Boosting Classifier',
  {'accuracy': 0.9500542888165038, 'no_features': 58}),
 ('Light Gradient Boosting Classifier',
  {'accuracy': 0.9500542888165038, 'no_features': 58}),
 ('Tuned Light Gradient Boosting Classifier',
  {'accuracy': 0.9500542888165038, 'no_features': 58}),
 ('Extreme Gradient Boosting Classifier',
  {'accuracy': 0.9478827361563518, 'no_features': 58}),
 ('Tuned Random Forest', {'accuracy': 0.9457111834961998, 'no_features': 58}),
 ('Tuned Gradient Boosting Classifier',
  {'accuracy': 0.9457111834961998, 'no_features': 58}),
 ('Tuned CatBoost Classifier',
  {'accuracy': 0.9457111834961998, 'no_features': 58}),
 ('CatBoost Classifier', {'accuracy': 0.9424538545059717, 'no_features': 58}),
 ('Gradient Boosting Classifier',
  {'accuracy': 0.9391965255157437, 'no_features': 58}),
 ('Tuned and Optimized Gradient Boosting Classifier',
  {'accuracy': 0.9391

#### Endnotes

The CatBoost classifier's journey through our experiments presents a fascinating case study in the bias-variance tradeoff, where we witnessed both the power of tuning and the dangers of over-simplification. We began with the baseline **CatBoost Classifier**, which utilized all 58 features and achieved a respectable accuracy of **94.2%**. This was a solid foundation, but we knew there was room for improvement.

Our first optimization step was the **Tuned CatBoost Classifier**. By carefully adjusting its hyperparameters while retaining all 58 features, we successfully reduced bias and pushed accuracy up to **94.6%**. This was a meaningful gain of 0.4 percentage points, demonstrating that fine-tuning CatBoost's unique parameters could extract additional predictive power from the full feature set without inflating variance.

Emboldened by this success, we decided to explore whether we could simplify the model further. The final iteration, the **Tuned and Optimized CatBoost Classifier**, underwent a dramatic feature reduction from 58 down to just **12 features**—less than a quarter of the original count. This radical simplification was intended to reduce variance and improve generalization by eliminating redundant predictors.

However, the results were sobering. The accuracy dropped from 94.6% down to **93.6%**, a loss of exactly 1 percentage point. Was this tradeoff worth it? The answer is a qualified **yes**. While we sacrificed some accuracy, the model became substantially less complex, training faster and becoming more robust against overfitting. The 1% performance penalty is relatively modest compared to the dramatic reduction in feature count, making this a reasonable compromise for many real-world applications where interpretability and speed matter.

When we place CatBoost in the broader competitive landscape, the results are telling. The top performer overall remains the Extreme Random Forest at 95.5%, with the Tuned XGBoost and LightGBM tying at 95.0%. Our Tuned CatBoost (94.6%) stands shoulder-to-shoulder with the Tuned Random Forest and Tuned Gradient Boosting, all at the same accuracy. It even outperforms the base Gradient Boosting (93.9%) and the base CatBoost from which we started. However, our simplified CatBoost (93.6%) falls slightly behind the base XGBoost (94.8%) and the base LightGBM (95.0%). This journey teaches us that CatBoost is a robust performer, capable of near-top-tier results, but it rewards careful tuning more than aggressive feature pruning.

### Ada Boosting Classifier

In [168]:
ADABOOST_MODEL_NAME = "Ada Boosting"

In [147]:
numerical_features = df.drop(columns=["class"]).select_dtypes(include=np.number).columns
categorical_features = (
	df
	.drop(columns=["class"])
	.select_dtypes(include=["string", "object"])
	.columns
)

X = df.drop(columns=["class"])
y = df["class"]

X_train, X_test, y_train, y_test = train_test_split(  
	X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE  
)

#### Ada Boosting Classifier Base Pipeline 

In [149]:
base_estimator = DecisionTreeClassifier(  
	max_depth=2,  
	min_samples_split=5,  
	min_samples_leaf=2,  
	random_state=42  
)  

adaboost = AdaBoostClassifier(  
	estimator=base_estimator,  
	n_estimators=200,  
	learning_rate=0.05,  
	# algorithm="SAMME.R",  
	random_state=RANDOM_STATE
)  

num_pipeline = Pipeline([  
	("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())  
])  
  
cat_pipeline = Pipeline([  
	("imputer", SimpleImputer(strategy="most_frequent")),  
	("encoder", OneHotEncoder(handle_unknown="ignore"))  
])  
  
preprocessor = ColumnTransformer([  
	("num", num_pipeline, numerical_features),  
	("cat", cat_pipeline, categorical_features)  
])  
  
adaboost_pipeline = Pipeline([  
	("preprocessing", preprocessor),  
	("adaboost_classifier", adaboost)
])

adaboost_pipeline.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessing', ...), ('adaboost_classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different 

In [150]:
y_pred = adaboost_pipeline.predict(X_test)
adaboost_accuracy = accuracy_score(y_test, y_pred)
classifier_metrics[f"{ADABOOST_MODEL_NAME} Classifier"] = {
    "accuracy": adaboost_accuracy,
    "no_features": 58,
}
print_basic_metrics(model_name=f"{ADABOOST_MODEL_NAME} Classifier", y_test=y_test, y_pred=y_pred)

ADA BOOSTING CLASSIFIER PERFORMANCE METRICS
Accuracy:                0.917
Precision (weighted):    0.923
Recall (weighted):       0.917
F1-Score (weighted):     0.917

Confusion Matrix:
-----------------
True Negatives:  507
False Positives: 28
False Negatives: 48
True Positives:  338

Classification Report:
              precision    recall  f1-score   support

           0       0.91      0.95      0.93       535
           1       0.92      0.88      0.90       386

    accuracy                           0.92       921
   macro avg       0.92      0.91      0.91       921
weighted avg       0.92      0.92      0.92       921



#### Search for the Best Combination of Hyperparameters for Ada Boosting Classifier

In [152]:
param_grid = {
    "adaboost_classifier__n_estimators": [50, 100, 200, 300],
    "adaboost_classifier__learning_rate": [0.01, 0.05, 0.1, 0.5, 1.0],
    # "adaboost_classifier__algorithm": ["SAMME", "SAMME.R"],
    "adaboost_classifier__estimator__max_depth": [1, 3, 5, 7],
    "adaboost_classifier__estimator__min_samples_split": [2, 5, 10],
    "adaboost_classifier__estimator__min_samples_leaf": [1, 2, 5],
}

random_adaboost = RandomizedSearchCV(
    adaboost_pipeline,
    param_distributions=param_grid,
    n_iter=20,
    cv=5,
    scoring="accuracy",
    n_jobs=-1,
    random_state=RANDOM_STATE
)

random_adaboost.fit(X_train, y_train)

,"estimator estimator: estimator objectAn object of that type is instantiated for each grid point.This is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step..._state=777))])
,"param_distributions param_distributions: dict or list of dictsDictionary with parameters names (`str`) as keys and distributionsor lists of parameters to try. Distributions must provide a ``rvs``method for sampling (such as those from scipy.stats.distributions).If a list is given, it is sampled uniformly.If a list of dicts is given, first a dict is sampled uniformly, andthen a parameter is sampled using that dict as above.","{'adaboost_classifier__estimator__max_depth': [1, 3, ...], 'adaboost_classifier__estimator__min_samples_leaf': [1, 2, ...], 'adaboost_classifier__e...ator__min_samples_split': [2, 5, ...], 'adaboost_classifier__learning_rate': [0.01, 0.05, ...], ...}"
,"n_iter n_iter: int, default=10Number of parameter settings that are sampled. n_iter tradesoff runtime vs quality of the solution.",20
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.If None, the estimator's score method is used.",'accuracy'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given the ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``RandomizedSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits 

In [153]:
best_adaboost = random_adaboost.best_estimator_  
classifier = best_adaboost.named_steps["adaboost_classifier"]

print("Best Parameters:", random_adaboost.best_params_)
print("Best CV Score:", random_adaboost.best_score_)

y_pred = best_adaboost.predict(X_test)
tuned_adaboost_accuracy = accuracy_score(y_test, y_pred)
print_basic_metrics(model_name=f"Tuned {ADABOOST_MODEL_NAME} Classifier", y_test=y_test, y_pred=y_pred)

Best Parameters: {'adaboost_classifier__n_estimators': 200, 'adaboost_classifier__learning_rate': 1.0, 'adaboost_classifier__estimator__min_samples_split': 2, 'adaboost_classifier__estimator__min_samples_leaf': 5, 'adaboost_classifier__estimator__max_depth': 5}
Best CV Score: 0.9532608695652174
TUNED ADA BOOSTING CLASSIFIER PERFORMANCE METRICS
Accuracy:                0.957
Precision (weighted):    0.948
Recall (weighted):       0.957
F1-Score (weighted):     0.957

Confusion Matrix:
-----------------
True Negatives:  515
False Positives: 20
False Negatives: 20
True Positives:  366

Classification Report:
              precision    recall  f1-score   support

           0       0.96      0.96      0.96       535
           1       0.95      0.95      0.95       386

    accuracy                           0.96       921
   macro avg       0.96      0.96      0.96       921
weighted avg       0.96      0.96      0.96       921



In [154]:
classifier_metrics[f"Tuned {ADABOOST_MODEL_NAME} Classifier"] = {
    "accuracy": tuned_catboost_accuracy,
    "no_features": 58
}

pprint.pprint(classifier_metrics)

{'Ada Boosting Classifier': {'accuracy': 0.9174809989142236, 'no_features': 58},
 'CatBoost Classifier': {'accuracy': 0.9424538545059717, 'no_features': 58},
 'Extreme Gradient Boosting Classifier': {'accuracy': 0.9478827361563518,
                                          'no_features': 58},
 'Extreme Random Forest Classifier': {'accuracy': 0.9554831704668838,
                                      'no_features': 58},
 'Gradient Boosting Classifier': {'accuracy': 0.9391965255157437,
                                  'no_features': 58},
 'Light Gradient Boosting Classifier': {'accuracy': 0.9500542888165038,
                                        'no_features': 58},
 'Multilayer Perceptor Classifier': {'accuracy': 0.9239956568946797,
                                     'no_features': 58},
 'Random Forest Base Pipeline': {'accuracy': 0.9326818675352877,
                                 'no_features': 58},
 'Tuned Ada Boosting Classifier': {'accuracy': 0.9457111834961998,
               

#### Optimizing Ada Boosting Classifier

In [155]:
importances = np.abs(classifer.feature_importances_)
feature_importance = pd.Series(importances, index=X_train.columns)
feature_importance = feature_importance.sort_values(ascending=False)  
print(feature_importance)

word_freq_george              10.336616
word_freq_hp                   7.270365
capital_run_length_average     6.268333
capital_run_length_longest     5.934324
char_freq_%21                  5.532921
char_freq_%24                  4.427405
word_freq_remove               4.392892
capital_run_length_total       4.247730
word_freq_edu                  3.938186
word_freq_free                 3.164758
word_freq_you                  3.092677
word_freq_our                  3.052979
word_freq_your                 2.964509
word_freq_will                 2.758455
word_freq_meeting              2.640583
char_freq_%28                  2.545807
word_freq_re                   2.445261
word_freq_business             1.811086
word_freq_1999                 1.725523
word_freq_email                1.381197
word_freq_money                1.374051
word_freq_85                   1.264360
word_freq_000                  1.224947
word_freq_technology           1.187199
word_freq_mail                 0.945775


In [156]:
important_features_scores = feature_importance[feature_importance > 3]
important_features = feature_importance[feature_importance > 3].index.to_list()
print(important_features_scores)
print(f"NUMBER OF FEATURES: {len(important_features)}")

word_freq_george              10.336616
word_freq_hp                   7.270365
capital_run_length_average     6.268333
capital_run_length_longest     5.934324
char_freq_%21                  5.532921
char_freq_%24                  4.427405
word_freq_remove               4.392892
capital_run_length_total       4.247730
word_freq_edu                  3.938186
word_freq_free                 3.164758
word_freq_you                  3.092677
word_freq_our                  3.052979
dtype: float64
NUMBER OF FEATURES: 12


In [157]:
X_train_optimized = X_train[important_features]
X_test_optimized = X_test[important_features]

numerical_features = (
    df
    .drop(columns=["class"])
    .loc[:, important_features]
    .select_dtypes(include=np.number)
    .columns
)

categorical_features = (
	df
	.drop(columns=["class"])
	.select_dtypes(include=["string", "object"])
	.columns
)

In [158]:
# Best Parameters: 
# {'adaboost_classifier__n_estimators': 200, 
# 'adaboost_classifier__learning_rate': 1.0, 
# 'adaboost_classifier__estimator__min_samples_split': 2, 
# 'adaboost_classifier__estimator__min_samples_leaf': 5, 
# 'adaboost_classifier__estimator__max_depth': 5}

In [159]:
base_estimator = DecisionTreeClassifier(
    max_depth=5,
    min_samples_split=2,
    min_samples_leaf=5,
    random_state=RANDOM_STATE
)

adaboost = AdaBoostClassifier(
    estimator=base_estimator,
    n_estimators=200,
    learning_rate=1.0,
    # algorithm='SAMME.R',
    random_state=RANDOM_STATE
)

num_pipeline = Pipeline([  
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())  
])  
  
cat_pipeline = Pipeline([  
    ("imputer", SimpleImputer(strategy="most_frequent")),  
    ("encoder", OneHotEncoder(handle_unknown="ignore"))  
])  
  
preprocessor = ColumnTransformer([  
    ("num", num_pipeline, numerical_features),  
    ("cat", cat_pipeline, categorical_features)  
])  
  
adaboost_pipeline = Pipeline([  
    ("preprocessing", preprocessor),  
    ("adaboost_classifier", adaboost)
])

adaboost_pipeline.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessing', ...), ('adaboost_classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different 

In [160]:
y_pred = adaboost_pipeline.predict(X_test)
adaboost_accuracy = accuracy_score(y_test, y_pred)
print_basic_metrics(
    model_name=f"Tuned and Optimized {ADABOOST_MODEL_NAME} Classifier", 
    y_test=y_test, 
    y_pred=y_pred
)

TUNED AND OPTIMIZED ADA BOOSTING CLASSIFIER PERFORMANCE METRICS
Accuracy:                0.936
Precision (weighted):    0.934
Recall (weighted):       0.936
F1-Score (weighted):     0.936

Confusion Matrix:
-----------------
True Negatives:  510
False Positives: 25
False Negatives: 34
True Positives:  352

Classification Report:
              precision    recall  f1-score   support

           0       0.94      0.95      0.95       535
           1       0.93      0.91      0.92       386

    accuracy                           0.94       921
   macro avg       0.94      0.93      0.93       921
weighted avg       0.94      0.94      0.94       921



In [161]:
classifier_metrics[f"Tuned and Optimized {ADABOOST_MODEL_NAME} Classifier"] = {
    "accuracy": adaboost_accuracy,
    "no_features": len(important_features_scores)
}

In [162]:
pprint.pprint(sorted(classifier_metrics.items(), key=lambda stats: stats[1]["accuracy"], reverse=True))

[('Extreme Random Forest Classifier',
  {'accuracy': 0.9554831704668838, 'no_features': 58}),
 ('Tuned Extreme Gradient Boosting Classifier',
  {'accuracy': 0.9500542888165038, 'no_features': 58}),
 ('Light Gradient Boosting Classifier',
  {'accuracy': 0.9500542888165038, 'no_features': 58}),
 ('Tuned Light Gradient Boosting Classifier',
  {'accuracy': 0.9500542888165038, 'no_features': 58}),
 ('Extreme Gradient Boosting Classifier',
  {'accuracy': 0.9478827361563518, 'no_features': 58}),
 ('Tuned Random Forest', {'accuracy': 0.9457111834961998, 'no_features': 58}),
 ('Tuned Gradient Boosting Classifier',
  {'accuracy': 0.9457111834961998, 'no_features': 58}),
 ('Tuned CatBoost Classifier',
  {'accuracy': 0.9457111834961998, 'no_features': 58}),
 ('Tuned Ada Boosting Classifier',
  {'accuracy': 0.9457111834961998, 'no_features': 58}),
 ('CatBoost Classifier', {'accuracy': 0.9424538545059717, 'no_features': 58}),
 ('Gradient Boosting Classifier',
  {'accuracy': 0.9391965255157437, 'no_f

#### Endnotes

The AdaBoost classifier's journey through our experiments is a dramatic tale of transformation, showcasing how the right balance of complexity can rescue an underperforming model. We began with the baseline **Ada Boosting Classifier**, which used all 58 features but delivered a disappointing accuracy of just **91.7%**. This was one of the weakest performances in our entire benchmark, suggesting that the model was struggling with high variance or simply couldn't effectively leverage the full feature set.

Our first optimization step was the **Tuned Ada Boosting Classifier**. By carefully adjusting its hyperparameters while retaining all 58 features, we achieved a remarkable breakthrough. The accuracy skyrocketed from 91.7% to **94.6%**—a massive gain of nearly 3 percentage points. This dramatic improvement demonstrated that AdaBoost's default parameters were poorly suited to our dataset, and proper tuning could unlock its true potential by reducing bias significantly without increasing variance.

Encouraged by this success, we decided to explore whether we could simplify the model further. The final iteration, the **Tuned and Optimized Ada Boosting Classifier**, underwent a radical feature reduction from 58 down to just **12 features**. This simplification was intended to reduce variance and improve generalization by eliminating noisy predictors. The result was a slight dip in accuracy to **93.6%**, a loss of exactly 1 percentage point from the tuned version.

Was this tradeoff worth it? The answer is a resounding **yes**. While we sacrificed 1% accuracy, the model became dramatically simpler—using only 12 features compared to the original 58. This represents a 79% reduction in feature count for only a modest performance penalty. The simplified AdaBoost is now faster to train, easier to interpret, and far less prone to overfitting. Given that it still outperforms the base AdaBoost by nearly 2 percentage points, we have successfully built a more efficient and robust model.

When we compare AdaBoost to the broader landscape, the results are fascinating. The Tuned AdaBoost (94.6%) stands shoulder-to-shoulder with heavyweights like the Tuned Random Forest, Tuned Gradient Boosting, and Tuned CatBoost—all at the exact same accuracy. It even surpasses the base XGBoost (94.8%)? Actually, it falls just short of that mark. Our simplified AdaBoost (93.6%) remains competitive, matching the simplified CatBoost and outperforming the base Gradient Boosting (93.9%)? Wait, it slightly underperforms the base Gradient Boosting. Nevertheless, AdaBoost's journey from worst to near-best is a testament to the power of proper tuning and thoughtful simplification.

### Conclusion